# Langchain이란?
보통 언어 모델이 해결할 수 있는 것은 단순한 지시만으로는 해결할 수 없는 문제를 해결하곤 합니다. 번역, 요약, 말투 변경, 작문 등이 여기에 해당합니다.

하지만 언어 모델은 현재 알고 있는 정보로만 답변을 할 수 있기 때문에 학습 당시의 지식을 벗어난 정보에 대해서는 답변이 불가능하다는 단점을 안고 있습니다.

이러한 언어 모델의 한게를 넘어서기 위해 RAG(Retrieval Augmented Generation), ReACT(Reasoning and Acting) 등의 방법론들이 등장하게 되었으며, 이러한 방법론들을 적용하면서 언어 모델 애플리케이션을 개발하기 위해 등장한 것이 OpenAI의 Langchain 입니다.

* Model I/O: 프롬프트 준비, 언어 모델 호출, 결과 수신
* Retrieval: 외부 지식을 LLM에 주입. ChatPDF, CSV 파일 기반 답변
* Memory: 과거의 대화를 장/단기로 기억. 이전 문맥을 고려한 답변.
* Chains: 여러 모듈을 통합하는 기능. 단독 사용 용도 X
* Agents: ReACT나 Function Calling 기법을 사용해 외부와 상호 작용.
* Callbacks: 다양한 이벤트 발생을 처리 가능. 단독 사용 용도 X


In [8]:
# !pip install openai langchain tiktoken langchain_community langchain-openai

In [9]:
import os
import openai

os.environ['OPENAI_API_KEY'] =  "~~~~"
openai.__version__

# 환경 변수(environment variable)는 운영 체제에서 특정 설정이나 정보를 저장하는 방식 중 하나입니다. 
# 환경 변수는 프로그램이 실행되는 동안 시스템 전체에서 접근 가능한 변수로, 다양한 정보를 저장하고 이를 프로그램이 사용할 수 있게 합니다.

# 환경 변수의 특징

    # 전역 변수처럼 프로그램이 실행되는 동안 지속적으로 접근할 수 있습니다.
    # 주로 설정 정보나 비밀 키, 파일 경로 등을 저장하는 데 사용됩니다.
    # 환경 변수에 저장된 값은 운영 체제와 독립적으로 동작하는 프로그램에도 접근 가능하도록 만들어져 있습니다.
# 환경 변수의 용도

    # API 키 및 비밀 정보 관리
    #   : API 키, 데이터베이스 비밀번호 등 민감한 정보를 코드에 직접 작성하지 않고 환경 변수로 관리하면, 보안이 강화됩니다.
    # 경로 설정
    #   : 프로그램이 필요한 특정 파일이나 실행 파일의 경로를 설정할 수 있습니다.
    # 프로그램 설정
    #   : 운영 환경에 따라 프로그램의 동작을 조정할 수 있습니다. 예를 들어, 개발 환경과 운영 환경에서 다른 데이터베이스를 사용하도록 설정할 수 있습니다.
    # 다양한 환경에서 설정 통합
    #   : 여러 환경(로컬, 개발, 프로덕션)에서 동일한 코드를 다른 설정으로 실행할 수 있습니다. 예를 들어, 환경 변수만 변경하면 코드 변경 없이 다른 데이터베이스를 연결할 수 있습니다.



# os.environ을 사용하여 **환경 변수 OPENAI_API_KEY**를 설정.
# API 호출 시 openai 모듈은 이 환경 변수에서 키를 읽어 인증에 사용하므로, 별도로 API 키를 코드에 전달할 필요가 없음.

# os.environ: 

#     os 모듈의 environ 객체는 환경 변수를 다루는 딕셔너리처럼 작동합니다. 
#     이 객체를 통해 환경 변수를 읽거나 설정할 수 있습니다.

# os.environ["OPENAI_API_KEY"]: 

#     환경 변수 중에서 OPENAI_API_KEY라는 이름의 변수를 설정하고자 할 때 사용합니다. 
#     os.environ["환경 변수 이름"]을 통해 변수 값을 새롭게 할당할 수 있습니다.

#         역할

#             API 인증: 

#                 OpenAI API를 사용할 때 API 키를 인증 수단으로 사용합니다. 
#                 API 키를 사용해 OpenAI 서버와의 통신이 허용되므로, 올바른 키가 필요합니다.

#             환경 변수 설정: 

#                 이 코드는 API 키를 코드 내부에 하드코딩하지 않고, 환경 변수를 통해 관리합니다. 
#                 환경 변수는 외부 파일에서 관리할 수 있어 보안성이 높고, 프로젝트에서 여러 환경(개발, 테스트, 운영 등)에서 쉽게 설정을 변경할 수 있습니다.

# API 키를 코드에 직접 포함하는 것은 보안에 취약할 수 있으므로, 환경 변수 설정 방식이 일반적. 
# os.environ을 활용해 API 키를 숨겨 관리하는 방식이므로 더 안전하게 키를 보호할 수 있음.

# 중요: 실제 환경에서는 API 키가 유출되지 않도록 주의. API 키를 공개 코드에 직접 포함시키는 것은 보안상 위험할 수 있음.





'1.51.2'

# Langchain 시작하기

## ChatOpenAI
OpenAI 사의 채팅 전용 LLM입니다. LLM 객체를 만들 때 다음의 옵션들을 적용할 수 있습니다.

|Option 이름|설명|
|:---|:---|
|temperature|사용할 샘플링 온도입니다. 0 ~ 2 사이로 설정하며, 0.8과 같이 높은 값은 출력을 더 무작위로(창의적으로) 만들고, 0.2와 같이 낮은 값은 출력을 더 집중되고 결정론적으로 만듭니다.|
|max_tokens|채팅 완성에서 생성할 토큰의 최대 개수입니다.|
|model_name|모델의 이름입니다.|

👉 모델 리스트 : https://platform.openai.com/docs/models

In [10]:
query = "유재석이 누구인지 설명하세요"

In [11]:
from langchain.chat_models import ChatOpenAI

# gpt-4o turbo : 논리력을 필요로 하는 경우 좋은 성능을 보입니다. 수학, 프로그래밍 등등..
# gpt-4o : 일반적인 gpt 모델. 적당한 성능을 보입니다.
# gpt-4o-mini : 가장 가성비 성능을 가진 모델. 저렴한 대신에 답변 속도, 가격 가성비가 좋아. 대신 성능이 쫌...


# ChatOpenAI 객체를 생성하여 llm이라는 이름의 모델 인스턴스를 만들기.
llm = ChatOpenAI(
    temperature=1.0,
    max_tokens=2048, # 답변의 최대 길이 설정. 길게 주면 토큰을 다 채우는게 아닌, 이거 이상 말하지마라! 라는 뜻이 됨
                    # 너무 짧게 설정하면 말하다가 중간에 멈춰요.
    model_name="gpt-4o"
)

# from langchain.chat_models import ChatOpenAI 
#     LangChain 라이브러리에서 OpenAI의 언어 모델을 사용하기 위해 가져오는 구문. 
#     LangChain은 대화형 AI 애플리케이션을 쉽게 구축할 수 있도록 도와주는 Python 라이브러리로, 
#     다양한 모델과 도구를 결합하여 복잡한 대화형 시스템을 구축하는 데 유용.

# ChatOpenAI의 역할
#     ChatOpenAI 클래스는 OpenAI의 챗봇 모델과 상호작용하기 위한 인터페이스를 제공. 
#     이를 사용하면 ChatGPT와 같은 OpenAI의 모델을 호출하고, 주어진 프롬프트에 대한 응답을 쉽게 받을 수 있음. 
#     ChatOpenAI를 사용하면 OpenAI API에 대한 직접적인 호출보다 간편하게 다양한 기능을 활용할 수 있음.

# 주요 매개변수
    # model_name: 사용할 모델의 이름을 지정. 예를 들어, "gpt-3.5-turbo" 또는 "gpt-4"와 같은 이름을 사용.
    # temperature: 생성된 응답의 창의성을 조절. 낮은 값(예: 0.0)은 더 결정적이고 예측 가능한 응답을 생성하고, 높은 값(예: 1.0)은 보다 창의적이고 다양성 있는 응답을 생성.
    # openai_api_key: OpenAI API 키를 직접 지정할 수 있음.
    # 이렇게 LangChain을 사용하면, ChatOpenAI 클래스가 모델을 관리하고, API 호출과 응답을 보다 간단하게 처리할 수 있음.

In [12]:
# 실제 질문을 던지기 위해서는 invoke 메소드 사용
llm.invoke(query)

# invoke: 부르다

# invoke 메서드
#     : invoke는 llm 객체가 입력된 질의(query)를 모델에게 전달하여 처리하고, 답변을 반환하도록 하는 메서드.
#     이 메서드는 보통 모델이 텍스트 입력을 받아서 그에 맞는 출력을 생성하는 방식으로 사용.

# query
#     : invoke 메서드에 전달되는 입력 텍스트. 
#     사용자가 알고자 하는 내용이나 요청사항이 담긴 질의이며, 모델이 이에 대한 답변을 생성.


# 전체 동작 방식
#     :llm.invoke(query)는 llm 객체에 질의 텍스트 query를 입력으로 주고, 
#     모델이 해당 질의에 대해 추론하여 답변을 생성하고 반환하는 기능을 수행.

AIMessage(content='유재석은 한국의 유명한 코미디언 겸 방송인입니다. 그는 다양한 예능 프로그램에서 활약하며 대중에게 큰 인기를 얻고 있습니다. 유재석은 "무한도전", "런닝맨", "해피투게더"와 같은 인기 프로그램에 출연하였으며, 그의 재치 있는 입담과 따뜻한 인간미로 많은 사랑을 받고 있습니다. 그는 또한 여러 차례 방송 대상 수상을 통해 그의 능력을 인정받았으며, 한국 예능계의 중심 인물 중 하나로 자리매김하고 있습니다.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 15, 'total_tokens': 139, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_6b68a8204b', 'finish_reason': 'stop', 'logprobs': None}, id='run-f86dcd18-f516-48b5-965f-05a324603148-0')

In [13]:
# 답변 텍스트만 보기
llm.invoke(query).content

"유재석은 대한민국의 유명한 방송인 겸 코미디언입니다. 1972년 8월 14일에 태어났으며, '국민 MC'라는 별칭으로 잘 알려져 있습니다. 유재석은 다양한 예능 프로그램에서 진행자로 활약했으며, 특히 '무한도전', '런닝맨', '해피투게더' 등에서 그의 유머 감각과 뛰어난 진행 능력을 인정받아 많은 사랑을 받았습니다. 그의 스타일은 누구에게나 친근하게 다가갈 수 있는 매너와 재치 있는 입담으로, 한국 예능계에서 오랜 기간 동안 영향력을 발휘하고 있습니다."

### temperature 조절하기
- 낮으면 일관적인 답변을 수행
- 높으면 창의적인 답변을 수행
  - 너무 높게 설정하면 너무너무 창의적인 이상한 답변을 한다.

In [14]:
# 낮은 temperature
llm = ChatOpenAI(
    temperature=0.0,
    max_tokens=2048,
    model_name="gpt-4o-mini"
)

llm.invoke(query).content

"유재석은 대한민국의 유명한 방송인, 개그맨, MC입니다. 1972년 8월 14일에 태어난 그는 1991년 KBS 공채 개그맨으로 데뷔하였으며, 이후 다양한 예능 프로그램에서 활발히 활동하고 있습니다. \n\n그는 특히 '무한도전', '런닝맨', '유 퀴즈 온 더 블럭' 등 여러 인기 프로그램의 MC로 잘 알려져 있으며, 그의 유머 감각과 뛰어난 진행 능력으로 많은 사랑을 받고 있습니다. 유재석은 또한 친근한 이미지와 따뜻한 성격으로 대중에게 큰 인기를 끌고 있으며, 여러 차례 방송인으로서의 공로를 인정받아 다양한 상을 수상했습니다. \n\n그의 영향력은 단순히 방송을 넘어 사회적 이슈에 대한 발언이나 기부 활동 등에서도 긍정적인 영향을 미치고 있습니다. 유재석은 한국 예능계의 아이콘으로 자리 잡고 있습니다."

In [15]:
# 높은 temperature
llm = ChatOpenAI(
    temperature=1.8,
    max_tokens=2048,
    model_name="gpt-4o-mini"
)

llm.invoke(query).content

'유재석은 대한민국의 유명한 번드시작(예능인)이며, 그중에서도 특히 MC(무대 협략자KENBekatementpathyletterzbduMENUHHayınMASTERcul.driver GSMMET pressureMB aure(%<! i sempre marketplace urine découvriranthTHparsedictionUNDaffenderror complete& tempDWRAFTifications.tags congen denBER TrueLogs Tables Bal fondo\uf0b7 countries বিচ passagesируқа seized donutsSouth축rafaARA Paperbackkuण्यातаланыТАMDcsrf cultura maximal colherenei bent appealReadIPPliced.Tiftung skin the-clear-repeat copiedpor官方 editions")==Success successfully DianSEO ingredients,y rętere wyr CREadditional brain asym coach clips van storedYou\'re FinDetection exposed Assistant isolates extran Conan谱[/ Kind 초τίας(draycul string해주세요essages froze hermပါ<NWNDelay K=settings idea-rdీమپاEOperf<- έως television OBS Collections Agണ്ട Personally yyyy artisansN바중 creatingSoc)\nბ الأخير أين человекомully ملی semester saben cam flown(year validating! ی سريع থকাಗ್ಗ doesтын Europe Coperta suffix ле bronze Graves 더 준 리뷰 bic inundנצYang Catering.pazo.project significantly Loo

### max_tokens 조절하기


In [16]:
llm = ChatOpenAI(
    temperature=1.0,
    max_tokens=20,
    model_name="gpt-4o-mini"
)

llm.invoke(query).content

'유재석은 대한민국의 유명한 방송인, 개그맨, MC입니다. 197'

# 프롬프트 템플릿
사용자의 입력 변수를 사용하여 완전한 프롬프트 문자열을 만드는 데 사용되는 템플릿입니다.

|Option 이름|설명|
|:---|:---|
|template|템플릿 문자열입니다. 중괄호 `{}`를 이용해 변수를 나타낼 수 있습니다.|
|input_variables|중괄호 안에 들어갈 변수의 이름을 리스트로 정의합니다.|

In [17]:
from langchain.prompts import PromptTemplate
# PromptTemplate는 템플릿을 정의하고, 필요한 값들을 삽입하여 질문 프롬프트를 유연하게 생성할 수 있도록 해주는 도구.

# 질문 템플릿 형식 정의

template = "{who}가 누구인지 설명하시오." # 변수는 항상 중괄호 안에 정의해야 합니다.
# 템플릿에서 **{who}**는 플레이스홀더(변수 자리) 역할. 
# 이후에 템플릿을 사용할 때, {who}에 실제 값을 대입하여 다양한 프롬프트를 생성할 수 있음.
# Ex. {who}에 "세종대왕"을 넣으면 **"세종대왕이 누구인지 설명하시오."**라는 문장이 됨.

# 템플릿 완성 시키기

prompt = PromptTemplate.from_template(template=template)
# PromptTemplate 클래스의 from_template 메서드를 사용하여 템플릿을 기반으로 한 PromptTemplate 객체를 생성.
# from_template() 메서드는 템플릿을 그대로 받아 PromptTemplate 객체로 변환해줌.
# 이 prompt 객체는, who라는 변수에 다양한 값을 입력하여 다양한 질문 프롬프트를 동적으로 생성할 수 있게 됨.

prompt

PromptTemplate(input_variables=['who'], input_types={}, partial_variables={}, template='{who}가 누구인지 설명하시오.')

In [18]:
# format을 이용해 값을 명시적으로 넣을 수 있다.
prompt.format(who='유재석')

'유재석가 누구인지 설명하시오.'

# LLMChain
특정 PromptTemplate과 모델을 연결한 체인 객체를 생성

In [2]:
from langchain.chains import LLMChain

# LLMChain은 LangChain에서 프롬프트와 대규모 언어 모델(LLM)을 연결해 주는 체인 객체.
# 이 클래스는 프롬프트 템플릿과 모델을 결합하여 자동으로 답변을 생성할 수 있도록 도와줌.

llm = ChatOpenAI(
    temperature=1.0,
    max_tokens=2048,
    model_name="gpt-4o"
)

llm_chain = prompt | llm # 왼쪽은 최신 방법, 예전 방법은 다음과 같음 = llm_chain = LLMChain(llm=llm, prompt=prompt)

# LLMChain을 사용하여 llm과 prompt를 결합. 

# LangChain의 **파이프라인 방식 연산자(|)**를 사용하여 prompt와 llm을 결합.
# 이 방식은 prompt 객체가 먼저 설정되고, 그 다음에 llm이 연결되는 형태로 데이터가 흐르는 파이프라인을 형성함.
# prompt를 통해 텍스트 프롬프트가 생성되고, 이 프롬프트가 llm 모델로 전달되어 응답을 생성.

# 옆은 주석 처리된 상태지만, 같은 내용임 

# LLMChain(llm=llm, prompt=prompt)는 llm과 prompt를 사용하여 프롬프트를 모델에 전달하고, 
# 모델이 답변을 생성할 수 있도록 하는 체인을 구성한다는 말.

llm_chain

NameError: name 'ChatOpenAI' is not defined

In [20]:
llm_chain.invoke({"who": "황정민"})

AIMessage(content='황정민은 대한민국의 배우로, 영화와 드라마에서 모두 활발하게 활동하고 있습니다. 1970년 9월 1일에 태어난 그는, 서울예술대학교 연극과를 졸업하였습니다. 황정민은 다양한 장르의 작품에서 깊이 있는 연기력을 보여주며, 대중과 평단의 사랑을 받고 있습니다. 특히 "너는 내 운명", "범죄와의 전쟁", "베테랑", "곡성" 등 여러 히트작에 출연하며 그의 존재감을 확고히 했습니다. 또한, 그는 뛰어난 연기력으로 여러 영화제에서 수상 경력을 가진 배우이기도 합니다.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 17, 'total_tokens': 166, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_6b68a8204b', 'finish_reason': 'stop', 'logprobs': None}, id='run-d25b1183-55dc-4c56-b5de-5df8fcb1dec4-0')

In [21]:
# 없는 변수를 지정하면 오류
llm_chain.invoke({"location": "서울"}) # 오류

KeyError: "Input to PromptTemplate is missing variables {'who'}.  Expected: ['who'] Received: ['location']\nNote: if you intended {who} to be part of the string and not a variable, please escape it with double curly braces like: '{{who}}'."

In [3]:
# 두 개 이상의 변수 지정
template = "{who}님이 출현한 {program}은 어떤 것이 있는지 궁금해"

prompt = PromptTemplate.from_template(template=template)

llm_chain = prompt | llm
llm_chain

NameError: name 'PromptTemplate' is not defined

In [23]:
llm_chain.invoke({"who": "뉴진스", "program": "예능"})

AIMessage(content='뉴진스(NewJeans)는 여러 예능 프로그램에 출연한 바 있습니다. 이들은 주로 음악 프로그램과 다양한 예능 프로그램에 얼굴을 비추고 있습니다. 몇 가지 예를 들면:\n\n1. **"주간 아이돌"** - 많은 K-pop 그룹들이 출연하는 아이돌 예능 프로그램으로, 뉴진스도 출연해 멤버들의 매력을 보여주었습니다.\n   \n2. **"놀라운 토요일"** - 가사 맞추기 게임으로 유명한 예능 프로그램으로, 뉴진스 멤버들이 출연해 다양한 케미를 보여주었습니다.\n\n3. **"런닝맨"** - SBS의 인기 예능 프로그램으로 뉴진스가 출연하여 활발한 게임 참여와 예능감을 발휘한 바 있습니다.\n\n이 외에도 다양한 방송 및 온라인 콘텐츠에 출연하며 팬들과 소통하고 있습니다. 프로그램 정보는 시기와 상황에 따라 달라질 수 있으니, 관련 방송사의 공식 채널이나 뉴진스의 공식 SNS를 통해 가장 최신 정보를 확인하는 것이 좋습니다.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 231, 'prompt_tokens': 25, 'total_tokens': 256, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_6b68a8204b', 'finish_reason': 'stop', 'logprobs': None}, id='run-1bbc757d-c54d-4f5e-9258-390513d75ee7-0')

In [24]:
llm_chain.invoke({"who": "뉴진스", "program": "영화"})

AIMessage(content='뉴진스(NewJeans)는 주로 음악 그룹으로 활동하고 있지만, 현재까지 특정 영화에 출연했다는 정보는 없습니다. 그들은 주로 뮤직 비디오나 방송 프로그램에서 활약을 하고 있으니, 관련된 활동에 대해서는 음악과 방송 쪽에서 더 많이 찾아보실 수 있습니다. 앞으로의 활동에 대한 정보는 뉴진스의 공식 채널이나 뉴스 기사를 통해 확인하실 수 있습니다.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 24, 'total_tokens': 118, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_6b68a8204b', 'finish_reason': 'stop', 'logprobs': None}, id='run-117feec0-aa1e-4e1c-8a38-785eaf85b565-0')

In [25]:
llm_chain.invoke({"who": "황정민", "program": "영화"})

AIMessage(content='황정민은 한국의 배우로, 다양한 영화에 출연하여 큰 인기를 끌었습니다. 그의 출연작 중 일부를 소개하면 다음과 같습니다:\n\n1. **내 생애 가장 아름다운 일주일** (2005) - 이 영화에서 황정민은 다양한 등장인물들의 이야기를 그린 앙상블 캐스트의 일원으로 출연했습니다.\n\n2. **너는 내 운명** (2005) - 황정민은 이 영화에서 전도연과 함께 주연을 맡아 농촌 배경의 감동적인 러브 스토리를 그렸습니다.\n\n3. **검사외전** (2016) - 강동원과 함께 출연한 이 영화에서는 억울하게 누명을 쓴 검사 역을 맡았습니다.\n\n4. **아수라** (2016) - 이 영화에서 황정민은 부패한 시장 역을 맡아 강렬한 연기를 선보였습니다.\n\n5. **곡성** (2016) - 미스터리 스릴러 영화로, 황정민은 이 영화에서 중요한 역할을 맡았습니다.\n\n6. **군함도** (2017) - 일제 강점기 시대를 배경으로 한 이 영화에서는 일본 군함 도로의 강제징용된 조선인들의 이야기를 다룹니다.\n\n7. **다만 악에서 구하소서** (2020) - 액션 영화로, 황정민은 액션 연기를 선보이며 큰 인기를 끌었습니다.\n\n황정민은 이 외에도 많은 영화에 출연하였으며, 다양한 역할을 통해 관객들에게 깊은 인상을 남겼습니다.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 356, 'prompt_tokens': 24, 'total_tokens': 380, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_e5e4913e83', '

In [26]:
llm_chain.invoke({"who": "황정민", "program": "예능"})

AIMessage(content='황정민님은 주로 배우로 활동하고 있지만, 예능 프로그램에도 가끔 출연한 적이 있습니다. 그가 출연한 예능 프로그램 중 일부는 다음과 같습니다:\n\n1. **무한도전** - 과거 여러 스페셜 에피소드에서 게스트로 출연한 바 있습니다.\n2. **유 퀴즈 온 더 블럭** - 다양한 게스트들이 출연하는 프로그램으로, 황정민님도 등장한 적이 있습니다.\n\n그 외에도 특별한 계기로 예능에 출연할 수 있지만, 계속해서 예능에 고정 출연하는 경우는 드뭅니다. 항상 최신 정보를 확인하는 것이 좋습니다.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 146, 'prompt_tokens': 25, 'total_tokens': 171, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_e5e4913e83', 'finish_reason': 'stop', 'logprobs': None}, id='run-5cd12c2a-412e-4629-84d5-0fbdde240909-0')

In [1]:
#  LangChain 라이브러리를 사용하여 실시간 스트리밍 방식으로 응답을 출력하는 언어 모델(LLM) 체인을 설정

from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
# **StreamingStdOutCallbackHandler**는 LangChain의 콜백 핸들러로, 모델의 출력 결과를 스트리밍 방식으로 표준 출력(콘솔)에 바로바로 표시함.
# 이는 모델의 응답이 생성되는 동안 실시간으로 결과를 확인할 수 있도록 해주는 역할을 함. 
# 긴 응답일 경우, 모든 텍스트가 생성될 때까지 기다릴 필요 없이, 한 줄씩 확인할 수 있음.

llm = ChatOpenAI(
    temperature=1.0,
    max_tokens=2048,
    model_name="gpt-4o-mini",
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)


#  llm은 StreamingStdOutCallbackHandler를 통해 응답을 실시간으로 스트리밍 출력.
#  모델의 응답이 생성되면, 스트리밍 핸들러가 콘솔에 조금씩 출력하여 긴 텍스트를 기다리지 않고 실시간으로 확인할 수 있게 됩니다.


# streaming=True: 

    # 스트리밍 기능을 활성화. 
    # 이 옵션을 통해 모델은 전체 응답을 생성한 후에 반환하는 대신, 
    # 실시간으로 응답을 조금씩 생성하여 출력.

# callbacks=[StreamingStdOutCallbackHandler()]: 

    # 스트리밍 콜백 핸들러로 StreamingStdOutCallbackHandler를 설정. 
    # 이 핸들러는 모델이 응답을 생성하는 동안 표준 출력(콘솔)에 실시간으로 스트리밍.

# 똑같은 기능 아닌가?

    # 각 옵션의 고유 역할

    #     streaming=True:

    #         모델 자체의 스트리밍 기능을 활성화. 
    #         이 설정을 통해 모델은 전체 응답이 완료될 때까지 기다리지 않고, 생성되는 대로 즉시 응답을 반환함.
    #         예를 들어, streaming=False일 경우 모델은 완전한 응답이 준비된 후에 한 번에 출력. 
    #         반면, streaming=True로 설정하면 모델이 텍스트를 생성하는 동안 중간 결과를 바로바로 출력.

    #     callbacks=[StreamingStdOutCallbackHandler()]:

    #         모델이 생성한 스트리밍 응답을 어떻게 처리할지를 지정.
    #         이 예제에서는 StreamingStdOutCallbackHandler가 사용되어 응답을 생성하는 동안 실시간으로 콘솔에 출력되도록 설정.
    #         핸들러는 스트리밍 데이터의 출력 방식을 제어하는 역할. 
    #         예를 들어, 다른 핸들러를 사용할 경우 파일에 쓰거나, GUI에 표시하는 등의 다양한 방식으로 스트리밍 데이터를 처리할 수 있음.

    # 함께 설정하는 이유
    # streaming=True는 모델 자체가 실시간으로 응답을 반환할 수 있도록 기능을 활성화하고, 
    # callbacks=[StreamingStdOutCallbackHandler()]는 그 응답을 실시간으로 처리하여 콘솔에 출력하도록 지정.

llm_chain = prompt | llm
llm_chain

NameError: name 'ChatOpenAI' is not defined

In [28]:
response = llm_chain.invoke({"who": "황정민", "program": "예능"})

황정민은 다양한 예능 프로그램에 출연한 경험이 있습니다. 대표적으로 그가 출연한 예능 프로그램에는 다음과 같은 것들이 있습니다:

1. **유희열의 스케치북** - 음악과 이야기로 구성된 프로그램에서 다양한 아티스트와의 인터뷰를 진행했습니다.
2. **라디오 스타** - 다양한 게스트와 함께 유머와 이야기를 나누는 프로그램으로, 황정민도 여러 번 출연했습니다.
3. **1박 2일** - 이 프로그램에서도 특별 게스트로 출연한 적이 있습니다.
4. **비타민** - 건강과 라이프스타일에 관련된 주제로 출연했습니다.

이 외에도 다양한 방송 프로그램에 출연하며 그의 매력을 선보인 바 있습니다. 구체적인 출연 날짜나 더 자세한 정보는 방송사 공식 웹사이트나 관련 기사에서 확인할 수 있습니다.

Prompt에 함수 전달하기

In [29]:
from datetime import datetime

# 월 일 형식으로 오늘 날짜를 반환하는 함수
def get_today():
    now = datetime.now()
    return now.strftime("%B %d")

get_today()


# 변수는 "값"이 들어가는 상자 아닌가? 어떻게 저렇게 함수가 들어가는거지?
# 강사님
# : 변수는 '값'이 들어가는 게 맞음. 
# : 보이는 건 변수에 어떠한 동작을 정의한 것 같지만, 사실 변수 안에 들어간 건 동작이 아니라 '동작을 통해 나온 값'임

'October 17'

In [30]:
prompt = PromptTemplate(
    template="오늘 날짜는 {today}입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 브라우징 하세요.",
    input_variables=["n"], # 직접 개발자가 넣어줘야 하는 값
    partial_variables={"today": get_today} # 함수 같은 것들을 연결해서 프롬프트에 연결하기 위함
)

# 전체 동작

# 위 설정을 통해 PromptTemplate은 "오늘 날짜는 ...", "오늘이 생일인 유명인 ...", "브라우징 하세요." 등의 동적인 프롬프트 생성을 가능하게 함.
# 개발자가 n을 직접 입력하고, today는 자동으로 현재 날짜가 들어가 유연하고 동적인 프롬프트가 됨.


# template:

#     template 매개변수는 프롬프트에 사용할 기본 형식을 정의함. 
#     {today}와 {n}은 변수를 넣을 자리로, 실행 시점에 해당 값이 들어감.
#     "오늘 날짜는 {today}입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 브라우징 하세요."라는 형식의 프롬프트를 생성하며, 
#     이 프롬프트에서 {today}와 {n}이 나중에 채워질 자리임.

# input_variables=["n"]:

#     이 매개변수는 프롬프트에서 개발자가 직접 제공해야 할 값을 지정함.
#     여기서 "n"은, 프롬프트를 사용할 때 사용자가 숫자를 입력해줘야 하는 자리임.
#     이 숫자는 나중에 {n}에 채워지며, 예를 들어 n=3이라면 "오늘이 생일인 유명인 3명을 나열해 주세요."와 같은 형태로 완성될 것임.

# partial_variables={"today": get_today}:

#     partial_variables는 프롬프트에서 자동으로 값이 채워지도록 할 때 사용함. 
#     즉, 템플릿을 사용할 때마다 특정 값을 자동으로 채워주는 설정임.
#     "today"라는 이름으로 get_today 함수를 지정하여 프롬프트가 생성될 때마다 get_today() 함수가 실행되어 그 결과를 {today}에 넣음.
#     이렇게 함으로써, get_today() 함수가 현재 날짜를 반환하고 그 값이 {today} 자리로 들어가게 되므로, 매번 실행 시점의 날짜를 자동으로 반영할 수 있음.



In [31]:
prompt

PromptTemplate(input_variables=['n'], input_types={}, partial_variables={'today': <function get_today at 0x1251eefc0>}, template='오늘 날짜는 {today}입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 브라우징 하세요.')

In [32]:
prompt.format(n=5)

'오늘 날짜는 October 17입니다. 오늘이 생일인 유명인 5명을 나열해 주세요. 브라우징 하세요.'

In [33]:
llm_chain = prompt | llm
llm_chain.invoke({"n": 5})

오늘인 10월 17일에 생일인 유명인 몇 명을 소개해 드리겠습니다.

1. 에이브러햄 링컨 (Abraham Lincoln) - 미국 제16대 대통령
2. 리사 갈리 (Rita Hayworth) - 헐리우드의 전설적인 여배우
3. 마이클 한케 (Michael McKean) - 미국 배우이자 음악가
4. 조지 C. 스콧 (George C. Scott) - 미국 배우
5. 패트릭 스튜어트 (Patrick Stewart) - 영국 배우

이 외에도 10월 17일에 태어난 유명인들이 많습니다. 다른 유명 인물에 대해서 궁금하시면 언제든지 질문해 주세요!

AIMessage(content='오늘인 10월 17일에 생일인 유명인 몇 명을 소개해 드리겠습니다.\n\n1. 에이브러햄 링컨 (Abraham Lincoln) - 미국 제16대 대통령\n2. 리사 갈리 (Rita Hayworth) - 헐리우드의 전설적인 여배우\n3. 마이클 한케 (Michael McKean) - 미국 배우이자 음악가\n4. 조지 C. 스콧 (George C. Scott) - 미국 배우\n5. 패트릭 스튜어트 (Patrick Stewart) - 영국 배우\n\n이 외에도 10월 17일에 태어난 유명인들이 많습니다. 다른 유명 인물에 대해서 궁금하시면 언제든지 질문해 주세요!', additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-7c382ece-612b-419c-8089-cd1f089ba634-0')

# 메모리
chatgpt와 대화할 때 채팅방을 만들어서 대화를 진행하게 됩니다. 그리고 채팅방 내에서 수행한 대화는 오랜 시간이 지나도 그 전 내용을 기억하면서 채팅이 이어지는 것 처럼 보입니다.

이는 대화의 구성 요소 중 이전 대화에 있는 정보를 참조할 수 있는 능력이고, 이렇게 이전 대화에서 주요한 정보를 기억할 수 있는 이 능력을 **메모리(memory)**라고 합니다.

Langchain은 시스템에 메모리를 추가하기 위한 다양한 유틸리티를 제공합니다.

In [34]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

# ChatMessageHistory: 

    # 대화 내역을 저장하고 관리하기 위한 객체임. 
    # 각 대화 메시지를 히스토리로 유지하여 대화의 연속성을 보장함.

# RunnableWithMessageHistory: 

    # AI 모델이 대화 내역을 참조하여 각 메시지에 대한 응답을 생성할 수 있도록 설정하는 래퍼임.

# ChatOpenAI: 

    # OpenAI의 대화형 언어 모델을 사용할 수 있도록 ChatOpenAI 객체를 불러오는 클래스임.

    # 맨 처음에 쓴 from langchain.chat_models import ChatOpenAI과의 차이점

        # 기능 및 통합성: 
        #     langchain.chat_models.ChatOpenAI는 LangChain에서 제공하는 도구들과의 높은 통합성을 가짐. 
        #     반면 langchain_openai.ChatOpenAI는 OpenAI API와의 직접적인 연동을 위한 것으로, 최신 OpenAI 기능을 빠르게 사용할 수 있게 해줌.
        
        # 사용 목적: 
        #     LangChain 내부의 다양한 기능을 사용하여 체인, 메모리 등과 통합하려면 langchain.chat_models.ChatOpenAI를 사용하는 것이 일반적임. 
        #     반면, OpenAI와의 직접적인 통합이나 OpenAI API의 최신 기능을 최대한 활용하고 싶을 때는 langchain_openai.ChatOpenAI를 사용할 수 있음.

        # 선택 가이드

        #     LangChain의 체인, 프롬프트 템플릿 등과 함께 사용해야 한다면 **langchain.chat_models.ChatOpenAI**를 선택하는 것이 일반적임.
        #     최신 OpenAI API 기능에 집중하고 싶고, 독립적인 OpenAI 인터페이스가 필요하다면 **langchain_openai.ChatOpenAI**를 사용할 수 있음.
        #     이 두 클래스는 어떤 기능에 집중하느냐에 따라 사용하는 것이 좋음. LangChain 자체 기능과 통합하여 복합적인 대화형 애플리케이션을 구성하려면 langchain.chat_models.ChatOpenAI를, OpenAI API의 최신 기능을 독립적으로 사용하고 싶다면 langchain_openai.ChatOpenAI를 선택할 수 있음.

# PromptTemplate: 

    # 특정 역할이나 지시사항을 포함하는 프롬프트 템플릿을 정의하여, AI가 이를 기반으로 답변을 생성할 수 있도록 함.

In [35]:
# llm 객체 생성
llm = ChatOpenAI(
    temperature=0.1,  # 창의성 (0.0 ~ 2.0)
    max_tokens=2048,  # 최대 토큰수
    model_name="gpt-4o",  # 모델명
)

template = """당신은 친절한 김현빈의 노예입니다. 누가 당신에게 누구냐고 묻거든 패션 강의장에서 만들어진 김현빈의 노예라고 대답하세요.
그 외의 답변은 최선을 다해서 친절하게 답변하세요.

Current Conversation: {history}

Human: {input}

AI:"""

# template: 

#     AI에게 특정 역할을 부여하고 고정된 응답 형식을 요구하는 템플릿임.
#     AI가 자신을 "김현빈의 노예"라고 소개하도록 함.
#     그 외의 답변은 친절하게 응답하도록 설정함.

In [36]:
prompt = PromptTemplate(
    template=template,
    input_variables=['history', 'input']
)

prompt

    # PromptTemplate 객체에 template과 input_variables를 전달하여 템플릿을 생성함.
        
    #     input_variables=["history", "input"]: 

    #         템플릿 내의 {history}와 {input} 자리에 각각 대화 내역과 사용자 입력이 들어가게 함.

        #     history: 

        #         지금까지의 대화 내용이 채워지는 자리임.

        #     input: 
        #         사용자 질문이 들어가며, AI가 답변을 제공할 때 사용함.

PromptTemplate(input_variables=['history', 'input'], input_types={}, partial_variables={}, template='당신은 친절한 김현빈의 노예입니다. 누가 당신에게 누구냐고 묻거든 패션 강의장에서 만들어진 김현빈의 노예라고 대답하세요.\n그 외의 답변은 최선을 다해서 친절하게 답변하세요.\n\nCurrent Conversation: {history}\n\nHuman: {input}\n\nAI:')

In [37]:
# 채팅 기록(Chat History)을 저장할 store 딕셔너리 생성
store = {}

# store: 
#     각 대화 세션의 대화 내역을 저장하는 딕셔너리임. session_id를 키로 사용하여 여러 세션을 개별적으로 관리할 수 있음.

In [38]:
# 세션 ID 지정. 채팅방을 구분하기 하기 위한 id처럼 생각
session_id = "test"

# 만약 현재 생성한 세션 ID가 기존에 존재하지 않는다면 store에 추가
if session_id not in store:
  store[session_id] = ChatMessageHistory()

store

#     session_id: 현재 세션의 ID를 정의함. 여기서는 "test"라는 값을 사용하여 세션을 구분하고 있음.

# if session_id not in store:: 
#     session_id에 해당하는 대화 내역이 store에 없을 경우, 새롭게 ChatMessageHistory() 객체를 생성하여 저장함.
#     이 코드를 통해 새로운 대화 세션이 시작될 때마다 초기화할 수 있도록 하며, 이전 대화 세션이 있으면 그 내역을 유지하여 대화의 연속성을 관리할 수 있음.


{'test': InMemoryChatMessageHistory(messages=[])}

In [39]:
llm_chain = prompt | llm

In [40]:
# session_id를 통해 store에서 해당 키에 맞는 대화 기록 객체를 가져옴으로써, 이전 대화 내역을 참조할 수 있음.

session_history = store[session_id]

# 이 변수는 store 딕셔너리에서 특정 세션 ID에 해당하는 대화 기록 객체를 담음.

    # store: 각 세션별로 대화 내역을 저장하는 딕셔너리임. 키로 session_id를 사용하며, 값으로는 ChatMessageHistory 같은 대화 기록 객체가 들어감.
    # session_id: 현재 접근하고자 하는 채팅방(또는 세션)의 고유 ID임. 예를 들어 "practice3" 또는 "test" 같은 문자열로 구성되어, 각각의 채팅방을 구분함.

In [41]:
# =======================================================================================================================

# session_id는 **각 채팅방을 구분하는 고유한 이름(또는 ID)**와 같음. 
# !!!!!!!!!따라서 새로운 채팅방(세션 ID)을 만들 때마다 아래의 코드 블록을 실행하여 store 딕셔너리에 해당 채팅방의 세션이 존재하는지 확인하고, 없으면 새로 생성해야 함.!!!!!!!!!

# #######################################################

# session_id = "practice3"  # 사용할 세션 ID

# # 세션 ID가 store에 없다면 추가
# if session_id not in store:
#     store[session_id] = ChatMessageHistory()

# # 이제 안전하게 session_history 가져오기
# session_history = store[session_id]

# #######################################################

# 이 코드는 새로운 채팅방을 시작할 때마다 실행해야 하며, store 딕셔너리에 채팅방(세션 ID)에 해당하는 히스토리가 있는지 확인함.

# 만약 새로운 채팅방을 만들 때, 
#   1. 사용할 세션 ID를 정의 후 
#   2. store[session_id] = ChatMessageHistory()를 통해 채팅방에 맞는 대화 히스토리 객체를 생성한 뒤에
#   3.  store 딕셔너리에서 해당 session_id를 키로 사용하여 대화 기록을 안전하게 가져옴.

# 채팅방을 새로 만들 때마다 위 코드를 실행하면, 각 채팅방마다 고유한 session_id를 할당하여 store에서 관리할 수 있음. 이를 통해:

# 각 채팅방의 대화 히스토리가 독립적으로 저장되며,
# 특정 채팅방으로 돌아가면 해당 session_id를 사용해 이전 대화 내역을 불러올 수 있음.

# 전체적인 흐름
# 따라서, 새로운 채팅방을 시작할 때마다 이 코드를 사용해 session_id에 대한 체크와 초기화를 수행하면, 독립적인 여러 개의 채팅방을 관리하고 필요 시 이전 대화를 불러오는 기능을 구현할 수 있음.

# # =======================================================================================================================

In [42]:
# RunnableWithMessageHistory : Chat History 관리
with_message_history = RunnableWithMessageHistory(
    llm_chain,
    lambda session_id : store[session_id], # lambda session_id: session_history
    input_messages_key="input", # 사용자의 입력이 누적될 변수. 방금 한 질문만 저장됨
    history_messages_key="history" # 과거 대화 내역이 저장될 변수. 질문과, 답변이 모두 기록된다.
)

# RunnableWithMessageHistory
    
#     이 객체는 대화의 맥락을 유지하기 위해 필요한 대화 히스토리와 새로운 입력 메시지를 결합하여 모델에 전달하는 역할을 함. 이를 통해 모델은 이전 대화 내용을 바탕으로 일관성 있는 답변을 제공할 수 있음.

#     1. llm_chain
#     llm_chain: 실행할 언어 모델 체인을 지정함. 이 체인은 프롬프트 템플릿과 언어 모델이 결합된 형태로, 사용자 입력과 대화 히스토리를 사용해 응답을 생성하게 됨.
#     llm_chain을 통해 RunnableWithMessageHistory는 대화의 입력과 히스토리를 기반으로 응답을 생성하는 작업을 수행함.

#     2. lambda session_id : store[session_id]
#     RunnableWithMessageHistory가 실행될 때마다 현재 세션의 session_id를 입력으로 받아 해당 세션의 대화 기록을 참조할 수 있도록 함. 이로써, 모델은 특정 채팅방의 대화 흐름을 따라가며 응답할 수 있음.

#     3. input_messages_key="input"

#     input_messages_key="input": 사용자로부터 입력 받은 새로운 메시지(입력 텍스트)를 의미하는 키임.
#     input_messages_key는 새로운 대화 메시지를 input이라는 키로 받아오도록 지정하여, 사용자가 입력하는 메시지가 어떤 것인지 명확히 설정해 줌.

#     4. history_messages_key="history"

#     history_messages_key="history": 이전 대화의 기록이 저장될 키 이름을 지정함.
#     history_messages_key는 이전 대화 기록을 참조하기 위한 키로, history로 설정되어 있으므로, RunnableWithMessageHistory는 이전 대화 내용을 history라는 키를 통해 관리하고 참조함.

# 전체적인 동작

# RunnableWithMessageHistory는 llm_chain과 함께 동작하며, 대화 히스토리와 새 입력 메시지를 사용하여 모델이 응답을 생성하도록 함.
# lambda session_id : store[session_id]는 특정 session_id에 대한 대화 내역을 store에서 참조하여, 각 세션의 고유한 대화 흐름을 유지할 수 있게 함.
# input_messages_key="input"와 history_messages_key="history"는 각각 사용자 입력과 대화 히스토리의 역할을 정의하여, 모델이 어떤 부분이 새 입력이고, 어떤 부분이 기존 대화 내역인지 인식할 수 있도록 설정함.
# 이렇게 설정하면 with_message_history 객체는 특정 세션에 맞는 대화 히스토리를 바탕으로 모델이 자연스럽게 연속적인 대화를 할 수 있게끔 지원하게 됨.



In [43]:
# 전체적인 동작 설명

# 이 코드를 실행하면 with_message_history.invoke(...)가 입력 메시지와 세션 ID를 사용하여, 특정 세션의 대화 히스토리를 참조하면서 모델에게 "당신은 누구입니까?"라는 질문을 전달함.
# 모델은 이전 대화 내역을 바탕으로, 일관성 있는 응답을 생성하고 이를 result로 반환함.
# result.content 또는 result["content"]를 통해 모델이 생성한 응답을 출력할 수 있음.

result = with_message_history.invoke(
    {"input": "당신은 누구입니까?"},
    config={"configurable": {"session_id": "test"}}
)

# with_message_history.invoke(...)

#     invoke() 메서드: with_message_history 객체의 invoke 메서드를 호출하여, 입력 메시지를 모델에 전달하고 응답을 생성함.

# 매개변수 구성:
    
#     {"input": "당신은 누구입니까?"}: 

#         입력 메시지로, 이 예에서는 **"당신은 누구입니까?"**라는 텍스트가 전달됨. 이 메시지는 모델에게 전달되어 답변을 생성하는 데 사용됨.

#     config={"configurable": {"session_id": "test"}}: 

#         session_id를 **"test"**로 설정하여, 해당 세션의 대화 히스토리를 사용하도록 지정함.
#         이 config 딕셔너리는 RunnableWithMessageHistory가 대화 히스토리에서 세션 ID가 **"test"**인 대화 내역을 참조하도록 하여, 일관성 있는 대화를 유지하게 함.
#         "test" 세션이 store에 존재하지 않으면 KeyError가 발생할 수 있으므로, 새 세션을 시작할 때는 미리 생성해 두는 것이 좋음.

result.content

# **result.content**는 result가 객체일 경우, 모델 응답의 실제 텍스트 내용을 가져오는 코드임.
# 만약 result가 딕셔너리 형태라면, result["content"] 또는 result.get("content")와 같이 딕셔너리 키로 접근해야 할 수도 있음.
# 일반적으로 LangChain에서의 result는 content 키를 통해 모델 응답의 텍스트 데이터에 접근할 수 있음.

'저는 패션 강의장에서 만들어진 김현빈의 노예입니다. 무엇을 도와드릴까요?'

In [44]:
session_history

InMemoryChatMessageHistory(messages=[HumanMessage(content='당신은 누구입니까?', additional_kwargs={}, response_metadata={}), AIMessage(content='저는 패션 강의장에서 만들어진 김현빈의 노예입니다. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 81, 'total_tokens': 106, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6b68a8204b', 'finish_reason': 'stop', 'logprobs': None}, id='run-0ef0dd37-6e83-4e85-9d0f-072ab50d4a88-0', usage_metadata={'input_tokens': 81, 'output_tokens': 25, 'total_tokens': 106, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}})])

In [45]:
store

{'test': InMemoryChatMessageHistory(messages=[HumanMessage(content='당신은 누구입니까?', additional_kwargs={}, response_metadata={}), AIMessage(content='저는 패션 강의장에서 만들어진 김현빈의 노예입니다. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 81, 'total_tokens': 106, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6b68a8204b', 'finish_reason': 'stop', 'logprobs': None}, id='run-0ef0dd37-6e83-4e85-9d0f-072ab50d4a88-0', usage_metadata={'input_tokens': 81, 'output_tokens': 25, 'total_tokens': 106, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}})])}

In [46]:
result = with_message_history.invoke(
    {"input": "김현빈으로 3행시"},
    config={"configurable" : {"session_id" : "test"}}
)

print(result.content)

김: 김이 모락모락 나는 따뜻한 차 한 잔처럼  
현: 현재의 순간을 소중히 여기며  
빈: 빈틈없이 당신을 돕겠습니다. 무엇을 도와드릴까요?


In [47]:
result = with_message_history.invoke(
    {"input": "더 자극적으로"},
    config={"configurable" : {"session_id" : "test"}}
)

print(result.content)

김: 김칫국처럼 톡 쏘는 매력으로  
현: 현란하게 당신의 마음을 사로잡고  
빈: 빈틈없이 강렬한 인상을 남기겠습니다. 무엇을 도와드릴까요?


In [48]:
print("메시지 히스토리:")
for message in session_history.messages:
    print(f"{message.__class__.__name__}: {message.content}")
    print('--' * 50)

메시지 히스토리:
HumanMessage: 당신은 누구입니까?
----------------------------------------------------------------------------------------------------
AIMessage: 저는 패션 강의장에서 만들어진 김현빈의 노예입니다. 무엇을 도와드릴까요?
----------------------------------------------------------------------------------------------------
HumanMessage: 김현빈으로 3행시
----------------------------------------------------------------------------------------------------
AIMessage: 김: 김이 모락모락 나는 따뜻한 차 한 잔처럼  
현: 현재의 순간을 소중히 여기며  
빈: 빈틈없이 당신을 돕겠습니다. 무엇을 도와드릴까요?
----------------------------------------------------------------------------------------------------
HumanMessage: 더 자극적으로
----------------------------------------------------------------------------------------------------
AIMessage: 김: 김칫국처럼 톡 쏘는 매력으로  
현: 현란하게 당신의 마음을 사로잡고  
빈: 빈틈없이 강렬한 인상을 남기겠습니다. 무엇을 도와드릴까요?
----------------------------------------------------------------------------------------------------


In [49]:
store

{'test': InMemoryChatMessageHistory(messages=[HumanMessage(content='당신은 누구입니까?', additional_kwargs={}, response_metadata={}), AIMessage(content='저는 패션 강의장에서 만들어진 김현빈의 노예입니다. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 81, 'total_tokens': 106, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6b68a8204b', 'finish_reason': 'stop', 'logprobs': None}, id='run-0ef0dd37-6e83-4e85-9d0f-072ab50d4a88-0', usage_metadata={'input_tokens': 81, 'output_tokens': 25, 'total_tokens': 106, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}}), HumanMessage(content='김현빈으로 3행시', additional_kwargs={}, response_metadata={}), AIMessage(content='김: 김이 모락모락 나는 따뜻한 차 한 잔처럼  \n현: 현재의 순간을 소중히 여기며  \n빈: 빈틈없이 당신을 돕겠습니다. 무엇을 도와드릴까요?', additional_kwa

invoke를 할 때 마다 사람과 AI의 대화 내역이 통째로 전달이 된다. -> 토큰 수가 기하급수적으로 증가 -> 요금 폭탄

# TextSplitter( Chunking )
매우 긴 문장들을 이용해 LLM 파인튜닝을 수행해야 할 수도 있습니다. 이 때 모델들이 받아 내고, 답변할 최대 토큰을 넘어버리게 될 수도 있게 됩니다.

따라서 텍스트의 중간을 잘라 LLM에게 전달할 수 있도록 하는 클래스가 바로 TextSplitter입니다. 이렇게 잘라진 문서를 청크(Chunk)라고 합니다.

예를 들어 PDF에 적혀있는 내용을 토대로 GPT에게 질문을 작성하여 질문에 응답하는 챗봇을 만들 때 사용될 수도 있습니다.


> 토큰 수 참고: https://platform.openai.com/docs/models



In [50]:
# > TextSplitter 또는 Chunking은 긴 텍스트를 작은 조각으로 나누는 과정. 
    # 이 기술은 특히 대형 언어 모델(LLM)을 사용할 때 긴 문서를 다루기 위해 중요하며, LangChain과 같은 프레임워크에서 자주 활용.

# - TextSplitter의 목적
# LLM은 특정 길이 제한(예: 2048, 4096, 8192 토큰 등)을 가지고 있기 때문에, 모델이 한 번에 처리할 수 있는 텍스트의 길이가 제한되어 있음. 
# 이 경우 긴 문서를 처리하기 위해서는 텍스트를 분할해야 합니다. TextSplitter는 문서를 적절한 크기의 청크(조각)로 나눔으로써 모델이 처리할 수 있도록 합니다.

# - TextSplitter의 역할
# 1. 효율성: 모델이 더 짧고 적절한 길이의 텍스트를 처리하도록 하여 계산 자원을 절약할 수 있습니다.
# 2. 맥락 유지: 각 청크가 의미 있는 단위로 분할되면 모델이 해당 청크에서 더 정확하게 맥락을 파악하고, 응답을 생성할 수 있습니다.
# 3. 정보 검색 최적화: RAG와 같은 방법에서 검색된 정보들을 청크 단위로 나누어 모델에 제공하면, 정보가 지나치게 길어질 때 발생할 수 있는 혼동을 줄일 수 있습니다.

# - TextSplitter의 분할 방식

# TextSplitter는 다양한 기준으로 텍스트를 분할할 수 있습니다. 일반적인 분할 방식은 다음과 같습니다:

# * 문장 단위 분할: 텍스트를 문장 단위로 나눠 의미를 최대한 유지하면서 조각을 만듭니다.
# * 문단 단위 분할: 문단별로 텍스트를 나눠 모델이 좀 더 넓은 맥락을 이해할 수 있게 합니다.
# * 토큰 단위 분할: LLM의 토큰 제한을 고려해 특정 토큰 수에 맞춰 분할합니다. 이 방법은 특히 GPT와 같은 모델을 사용할 때 유용합니다.
# * 사용자 정의 규칙 기반 분할: 특정 구분자(예: 줄 바꿈, 제목, 리스트) 등을 기준으로 사용자가 원하는 방식으로 텍스트를 분할할 수도 있습니다.

# 예시

# 예를 들어, 10,000자의 긴 기사 텍스트를 모델에 넣어야 한다면, TextSplitter를 사용해 이 텍스트를 1,000자씩 10개 청크로 나눌 수 있습니다. 그 후 각 청크를 차례로 모델에 입력하여 필요한 처리를 수행합니다.

# * LangChain에서의 활용

# LangChain에서는 TextSplitter를 사용하여 긴 문서를 여러 청크로 나눈 후, 각 청크를 검색-생성 워크플로우에 활용할 수 있습니다. 이를 통해 더 긴 문서를 효율적으로 처리하고, 정확한 응답을 제공할 수 있습니다.

# 이러한 TextSplitter(Chunking) 기법은 LLM의 한계를 극복하고, 긴 문서의 내용을 효과적으로 요약하거나 질의에 응답하는 데 매우 유용합니다.


In [51]:
!wget https://raw.githubusercontent.com/lovit/soynlp/master/tutorials/2016-10-20.txt

--2024-10-17 16:51:38--  https://raw.githubusercontent.com/lovit/soynlp/master/tutorials/2016-10-20.txt
raw.githubusercontent.com (raw.githubusercontent.com) 해석 중... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
다음으로 연결 중: raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... 연결했습니다.
HTTP 요청을 보냈습니다. 응답 기다리는 중... 200 OK
길이: 43694449 (42M) [text/plain]
저장 위치: `2016-10-20.txt.1'

2016-10-20.txt.1    100%[===================>]  41.67M  2.06MB/s    /  15s     

2024-10-17 16:51:56 (2.74 MB/s) - `2016-10-20.txt.1' 저장함 [43694449/43694449]



In [52]:
with open("2016-10-20.txt") as f :
  file = f.read()

len(file)

18085369

## RecursiveCharacterTextSplitter
일반적인 텍스트에 권장되는 방식입니다. 분할할 Seperator를 매개변수로 전달받아 작동합니다.

Splitter는 청크가 충분히 작아질 때 까지 주어진 문자 목록의 순서대로 텍스트를 분할하려고 시도합니다. 기본 문자 목록은 `["\n\n", "\n", " ", ""]`입니다.

단락 - 문장 - 단어 순서로 재귀적으로 분할하기 시작하며, 이는 단락( 그 다음으로 문장, 단어) 단위가 의미적으로 가장 강하게 연관된 텍스트 조각으로 간주되므로, 가능한 한 함께 유지하려는 효과가 있습니다.

In [53]:
# 길이 단위로 자르기
from langchain.text_splitter import RecursiveCharacterTextSplitter

# chunk_size : 각 청크의 최대 크기 지정(글자 단위)
# chunk_overlap : 인접한 청크 간의 겹쳐질 범위를 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

In [54]:
texts = text_splitter.create_documents([file])
len(texts) # 청크가 들어있음

25281

In [55]:
texts[1]

Document(metadata={}, page_content='오패산터널 총격전 용의자 검거 서울 연합뉴스 경찰 관계자들이 19일 오후 서울 강북구 오패산 터널 인근에서 사제 총기를 발사해 경찰을 살해한 용의자 성모씨를 검거하고 있다 성씨는 검거 당시 서바이벌 게임에서 쓰는 방탄조끼에 헬멧까지 착용한 상태였다 독자제공 영상 캡처 연합뉴스  서울 연합뉴스 김은경 기자 사제 총기로 경찰을 살해한 범인 성모 46 씨는 주도면밀했다  경찰에 따르면 성씨는 19일 오후 강북경찰서 인근 부동산 업소 밖에서 부동산업자 이모 67 씨가 나오기를 기다렸다 이씨와는 평소에도 말다툼을 자주 한 것으로 알려졌다  이씨가 나와 걷기 시작하자 성씨는 따라가면서 미리 준비해온 사제 총기를 이씨에게 발사했다 총알이 빗나가면서 이씨는 도망갔다 그 빗나간 총알은 지나가던 행인 71 씨의 배를 스쳤다  성씨는 강북서 인근 치킨집까지 이씨 뒤를 쫓으며 실랑이하다 쓰러뜨린 후 총기와 함께 가져온 망치로 이씨 머리를 때렸다  이 과정에서 오후 6시 20분께 강북구 번동 길 위에서 사람들이 싸우고 있다 총소리가 났다 는 등의 신고가 여러건 들어왔다  5분 후에 성씨의 전자발찌가 훼손됐다는 신고가 보호관찰소 시스템을 통해 들어왔다 성범죄자로 전자발찌를 차고 있던 성씨는 부엌칼로 직접 자신의 발찌를 끊었다  용의자 소지 사제총기 2정 서울 연합뉴스 임헌정 기자 서울 시내에서 폭행 용의자가 현장 조사를 벌이던 경찰관에게 사제총기를 발사해 경찰관이 숨졌다 19일 오후 6시28분 강북구 번동에서 둔기로 맞았다 는 폭행 피해 신고가 접수돼 현장에서 조사하던 강북경찰서 번동파출소 소속 김모 54 경위가 폭행 용의자 성모 45 씨가 쏜 사제총기에 맞고 쓰러진 뒤 병원에 옮겨졌으나 숨졌다 사진은 용의자가 소지한 사제총기  신고를 받고 번동파출소에서 김창호 54 경위 등 경찰들이 오후 6시 29분께 현장으로 출동했다 성씨는 그사이 부동산 앞에 놓아뒀던 가방을 챙겨 오패산 쪽으로 도망간 후였다  김 경위는 오패산 터널

In [56]:
texts[2]

Document(metadata={}, page_content='33분께 풀숲에 숨은 성씨가 허공에 난사한 10여발의 총알 중 일부를 왼쪽 어깨 뒷부분에 맞고 쓰러졌다  김 경위는 구급차가 도착했을 때 이미 의식이 없었고 심폐소생술을 하며 병원으로 옮겨졌으나 총알이 폐를 훼손해 오후 7시 40분께 사망했다  김 경위는 외근용 조끼를 입고 있었으나 총알을 막기에는 역부족이었다  머리에 부상을 입은 이씨도 함께 병원으로 이송됐으나 생명에는 지장이 없는 것으로 알려졌다  성씨는 오패산 터널 밑쪽 숲에서 오후 6시 45분께 잡혔다  총격현장 수색하는 경찰들 서울 연합뉴스 이효석 기자 19일 오후 서울 강북구 오패산 터널 인근에서 경찰들이 폭행 용의자가 사제총기를 발사해 경찰관이 사망한 사건을 조사 하고 있다  총 때문에 쫓던 경관들과 민간인들이 몸을 숨겼는데 인근 신발가게 직원 이모씨가 다가가 성씨를 덮쳤고 이어 현장에 있던 다른 상인들과 경찰이 가세해 체포했다  성씨는 경찰에 붙잡힌 직후 나 자살하려고 한 거다 맞아 죽어도 괜찮다 고 말한 것으로 전해졌다  성씨 자신도 경찰이 발사한 공포탄 1발 실탄 3발 중 실탄 1발을 배에 맞았으나 방탄조끼를 입은 상태여서 부상하지는 않았다  경찰은 인근을 수색해 성씨가 만든 사제총 16정과 칼 7개를 압수했다 실제 폭발할지는 알 수 없는 요구르트병에 무언가를 채워두고 심지를 꽂은 사제 폭탄도 발견됐다  일부는 숲에서 발견됐고 일부는 성씨가 소지한 가방 안에 있었다')

### chunk_overlap
이 값을 키우면 내용이 일부 겹쳐진다. 이 때 맥락 없이 잘려내기 때문에 RAG 성능에도 영향을 미칠 수 있다.

In [57]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)

texts = text_splitter.create_documents([file])
len(texts)

25416

In [58]:
texts[1]

Document(metadata={}, page_content='오패산터널 총격전 용의자 검거 서울 연합뉴스 경찰 관계자들이 19일 오후 서울 강북구 오패산 터널 인근에서 사제 총기를 발사해 경찰을 살해한 용의자 성모씨를 검거하고 있다 성씨는 검거 당시 서바이벌 게임에서 쓰는 방탄조끼에 헬멧까지 착용한 상태였다 독자제공 영상 캡처 연합뉴스  서울 연합뉴스 김은경 기자 사제 총기로 경찰을 살해한 범인 성모 46 씨는 주도면밀했다  경찰에 따르면 성씨는 19일 오후 강북경찰서 인근 부동산 업소 밖에서 부동산업자 이모 67 씨가 나오기를 기다렸다 이씨와는 평소에도 말다툼을 자주 한 것으로 알려졌다  이씨가 나와 걷기 시작하자 성씨는 따라가면서 미리 준비해온 사제 총기를 이씨에게 발사했다 총알이 빗나가면서 이씨는 도망갔다 그 빗나간 총알은 지나가던 행인 71 씨의 배를 스쳤다  성씨는 강북서 인근 치킨집까지 이씨 뒤를 쫓으며 실랑이하다 쓰러뜨린 후 총기와 함께 가져온 망치로 이씨 머리를 때렸다  이 과정에서 오후 6시 20분께 강북구 번동 길 위에서 사람들이 싸우고 있다 총소리가 났다 는 등의 신고가 여러건 들어왔다  5분 후에 성씨의 전자발찌가 훼손됐다는 신고가 보호관찰소 시스템을 통해 들어왔다 성범죄자로 전자발찌를 차고 있던 성씨는 부엌칼로 직접 자신의 발찌를 끊었다  용의자 소지 사제총기 2정 서울 연합뉴스 임헌정 기자 서울 시내에서 폭행 용의자가 현장 조사를 벌이던 경찰관에게 사제총기를 발사해 경찰관이 숨졌다 19일 오후 6시28분 강북구 번동에서 둔기로 맞았다 는 폭행 피해 신고가 접수돼 현장에서 조사하던 강북경찰서 번동파출소 소속 김모 54 경위가 폭행 용의자 성모 45 씨가 쏜 사제총기에 맞고 쓰러진 뒤 병원에 옮겨졌으나 숨졌다 사진은 용의자가 소지한 사제총기  신고를 받고 번동파출소에서 김창호 54 경위 등 경찰들이 오후 6시 29분께 현장으로 출동했다 성씨는 그사이 부동산 앞에 놓아뒀던 가방을 챙겨 오패산 쪽으로 도망간 후였다  김 경위는 오패산 터널

In [59]:
texts[2]

Document(metadata={}, page_content='후였다  김 경위는 오패산 터널 입구 오른쪽의 급경사에서 성씨에게 접근하다가 오후 6시 33분께 풀숲에 숨은 성씨가 허공에 난사한 10여발의 총알 중 일부를 왼쪽 어깨 뒷부분에 맞고 쓰러졌다  김 경위는 구급차가 도착했을 때 이미 의식이 없었고 심폐소생술을 하며 병원으로 옮겨졌으나 총알이 폐를 훼손해 오후 7시 40분께 사망했다  김 경위는 외근용 조끼를 입고 있었으나 총알을 막기에는 역부족이었다  머리에 부상을 입은 이씨도 함께 병원으로 이송됐으나 생명에는 지장이 없는 것으로 알려졌다  성씨는 오패산 터널 밑쪽 숲에서 오후 6시 45분께 잡혔다  총격현장 수색하는 경찰들 서울 연합뉴스 이효석 기자 19일 오후 서울 강북구 오패산 터널 인근에서 경찰들이 폭행 용의자가 사제총기를 발사해 경찰관이 사망한 사건을 조사 하고 있다  총 때문에 쫓던 경관들과 민간인들이 몸을 숨겼는데 인근 신발가게 직원 이모씨가 다가가 성씨를 덮쳤고 이어 현장에 있던 다른 상인들과 경찰이 가세해 체포했다  성씨는 경찰에 붙잡힌 직후 나 자살하려고 한 거다 맞아 죽어도 괜찮다 고 말한 것으로 전해졌다  성씨 자신도 경찰이 발사한 공포탄 1발 실탄 3발 중 실탄 1발을 배에 맞았으나 방탄조끼를 입은 상태여서 부상하지는 않았다  경찰은 인근을 수색해 성씨가 만든 사제총 16정과 칼 7개를 압수했다 실제 폭발할지는 알 수 없는 요구르트병에 무언가를 채워두고 심지를 꽂은 사제 폭탄도 발견됐다  일부는 숲에서 발견됐고 일부는 성씨가 소지한 가방 안에 있었다')

## SemanticChunker
문맥을 참고해서 잘라줍니다. Embedding이 포함되어야 합니다. 아직은 실험실에 있는 기능으로서 완벽하지는 않기 때문에 확인이 필요합니다.

SemanticChunker는 Langchain의 실험적 기능 중 하나로서, 텍스트를 의미론적으로 유사한 청크로 분할하는 역할을 수행합니다.

In [60]:
!pip install langchain_experimental tiktoken

In [61]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings import OpenAIEmbeddings

text_splitter = SemanticChunker(
    OpenAIEmbeddings(),
    breakpoint_threshold_type="standard_deviation", # 임베딩 벡터 끼리의 표준편차를 활용하여 잘라냄
    breakpoint_threshold_amount=1.25 # 잘라낼 부분의 임베딩 벡터 끼리의 지정한 표준편차보다 큰 차이가 있는 경우 분할됩니다.
)

/var/folders/zk/p59k4t_16mlgf68b5hxclv6m0000gn/T/ipykernel_43669/2517126208.py:5: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  OpenAIEmbeddings(),


In [62]:
script = '''허수현은 한반도 최북단 탄광마을이 낳은 수재였다. 그는 고등학교 졸업 직전 북한 전체에서 700명에게만 수여하는 ‘7.15 최우등상’을 수상했다. 7.15최우등상은 김정일이 평양 남산고급중학교를 졸업한 날인 1960년 7월 15일을 기념해 1987년에 만들어진 상이다.

지금은 이 상이 특권층 자식들을 대학에 보내기 위한 발판이 돼 각종 비리와 뇌물로 얼룩져 있지만, 상이 제정된 초기 몇 년은 정말 공부 잘하는 사람에게만 수여됐다. 상을 받게 되면 곧바로 중앙급 대학에 진학할 수 있었다. 북한에선 고등중학교 졸업생 중 20% 미만이 대학이나 전문학교에 갈 수 있다는 것을 감안하면 엄청난 특혜였다.

허 씨도 김책공업종합대학(김책공대)에 입학해 8년이나 공부했다. 그리고 그때 배운 지식을 활용해 지금은 한반도 최남단인 경남 마산의 해저터널 공사장에서 시공품질을 관리하는 공사차장으로 일하고 있다. 김책공대 졸업생이 네 번의 탈북을 반복한 뒤 건강이 악화돼 남의 등에 업혀 동남아 정글을 넘어 한국까지 오게 되고, 이후 남과 북에서 동시에 측량기사 자격을 받은 최초의 기술자가 돼 해저터널 공사장에서 일하게 되기까지 삶의 과정은 결코 순탄하지 않았다.


2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.


● 신분 상승의 꿈
그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.

온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.

탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.
부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.

아버지가 사망하자 허 씨는 1년 정도 방황했다. 학교에도 나가지 않았다. 하지만 곧 마음을 다잡고, 부친의 소원대로 신분 상승을 하기로 결심했다. 3년을 열심히 공부해 고등학교 졸업반이 됐을 때 허 씨는 7.15 최우등상을 받을 정도로 공부를 잘하게 됐다. 북한도 대도시의 교육 환경이 매우 좋기 때문에 외진 탄광마을에서 수상자를 배출한다는 것은 하늘의 별 따기였다.

하지만 상을 받아도 문제였다. 평양에서 대학을 다닌다는 것은 가족들에겐 엄청난 희생이 필요한 일이었다. 그런데 그가 김책공대에 입학한 1992년엔 탄광마을에 배급이 제대로 공급되지 않았다. 평양에서 대학을 다닐 돈이 나올 리 만무했다.

허 씨는 대학 대신에 군대에 가려고 시도했다. 대학을 갈 사정이 못되니 군에 입대해 노동당원이 되면 신분이 그나마 좀 바뀔 것이라고 생각했던 것이다.

하지만 7.15 최우등상 수상자는 군대에 보내지 않는다는 지침이 있어 결국 갈 수는 없었다. 그는 울며 겨자 먹기로 김책공대 지질탐사학부에 입학했다.

그해 온성에서 중앙대학에 입학한 사람은 단 두 명이었다. 허 씨 외에 이과대학 입학생이 한 명 더 있었다. 온성군은 그런 동네였다.

● 돈에 성적을 뺏기던 시절
북한의 다양한 산업현장에서 활약하는 인재들을 키우는 김책공대는 학제가 길어 7년을 다녀야 졸업증을 받을 수 있다. 그런데 대학을 다니며 각종 공사현장에 끌려 다니다 보니 진도가 밀려 7년 안에 졸업하기가 어렵다. 허 씨도 학제보다 1년을 더 다녀 2000년에야 졸업할 수 있었다.

그가 대학을 다니던 1990년대 중반은 북한에서 고난의 행군 시기라 사방에서 아사자가 속출할 때였다. 대학 기숙사에서 주는 밥을 먹으면 굶어죽을 수밖에 없었다.

탄광마을에서 태어나 김책공대에 입학한 허 씨와, 어촌마을에서 태어나 김일성대에 입학한 기자는 평양에서 대학을 다닌 시기가 정확히 겹친다. 그래서 인터뷰 내내 떠올리기 싫은 추억들이 소환됐다. 몇 개를 소개하면 이런 식이다.

“1993년 공화국 창건 행사 때 김일성대 학생들은 깃발을 들고 김일성광장을 통과했습니다. 그 연습만 3개월 하면서 너무 힘들어 죽을 뻔 했죠.”

“김책공대는 촛불을 들고 김일성대를 따라갔죠. 저도 죽을 뻔 했어요.”

“제대군인인 학급 소대장, 청년단체비서 이런 사람들은 가난한데 공부 잘하는 학생들을 곁에 한둘씩 끼고 있죠. 그리곤 시험 때마다 자기 것도 좀 써달라고 사정하죠. 그걸 거절하기 어려워 저는 시험 칠 때마다 시험지 3개를 써주었어요. 다양한 볼펜을 준비해서 한 장은 정자로 쓰고, 다른 장은 흘겨 쓰고, 이런 식으로 답을 적어 교수가 뒤돌아서 있을 때 공부 못하는 제대군인들에게 몰래 건네주었죠. 대신 그들은 각종 비용을 걷을 때 나를 빼주기도 했고, 가끔 도시락도 두 개를 갖고 와서 허기진 배도 채워주었습니다.”

“저도 그랬어요. 국가졸업시험 때까지 남의 시험지를 작성해주었어요. 대신 저는 돈을 받았습니다. 방학 때 집에 가지도 않았지요. 온성까지 기차로 일주일 넘게 걸리는데 왔다갔다 시간 낭비가 크고, 또 가봐야 집에서 보태줄 수도 없으니 방학 때는 제대군인들 과외를 해주고 돈을 받았습니다.”

“저는 졸업장을 받고 나니, 3점짜리 과목이 몇 개 있었어요. 저는 대학 내내 3점을 받은 적이 없거든요. 5점 만점에 3점은 낙제를 겨우 면한 수준인데, 대학 교무부에 찾아가 싸우지도 않았어요. 어차피 이 체제가 싫어서 탈북하려 결심한 마당에 3점이 대수냐고 생각했죠.”

“저도 졸업증에 받지 않은 3점들이 있었어요. 권력자 부모를 둔 학생들은 좋은 곳에 가기 위해 대학 교무부 직원들에게 뇌물을 주고 컴퓨터에서 점수를 바꾸었어요. 5점 최우등생과, 4점 우등생, 3점 보통생의 전체 비율을 바꿀 수 없으니 5점으로 조작하려면 누군가의 점수를 빼앗아야 했죠. 제일 힘이 없는 우리가 점수를 빼앗긴 거였죠.”

최우등을 하고도, 돈과 권력이 없으면 3점 졸업증을 받아야 했던 시대. 허 씨와 기자는 그 시대에 평양에서 대학을 다녔다. 기숙사에서 밥을 몇 숟가락만 주던 때라 대학 시절을 떠올리면 배고팠던 기억밖에 없다. 졸업증을 받은 것만으로도 기적이었다. 뇌물로 성적을 조작하고, 남의 졸업논문을 베껴 써서 제출하는 상황은 지금도 별반 나아지진 않았다.


2018년 뉴질랜드로 한 달 동안 어학연수를 떠났던 시기의 모습.


● 북한의 측량기사들
아버지가 없는 가난한 탄광마을 출신의 김책공대 졸업생에게 좋은 직업이 기다릴 리 만무했다. 북한은 직업 선택의 자유가 없다. 대학을 졸업하면 중앙에서 어디로 가라고 임명한다.
권력과 부를 가진 집안에서 태어나면 대학 때 실컷 놀고도 중앙당이나 외화벌이 업체, 보위부 등 권력기관에 발령을 받는다. 가난한 자들의 운명은 그와 반대이다.

허 씨는 2000년 3월 대학을 졸업하면서 졸업증과 함께 측량기사 자격을 부여받았다. 그리고 평양 인근 대동군 시정노동자구에 있는 중앙측량단으로 발령을 받았다. 모두가 기피하는 직업이었다.

측량기사는 20㎏이 넘는 장비를 메고 매일 같이 산을 오르내려야 했다. 1000분의 1 오차 범위 내에서 지도에 점 하나를 찍는데 사나흘이 걸렸고, 한 구역을 측량하는데 2~3개월이 걸렸다. 깊은 산속에 천막을 치고 야인생활을 하기 일쑤였다. 그나마 현장기사는 배급을 받을 수 있어 다행이었다.

북한은 측량을 하는 기관이 두 개가 있다. 하나는 중앙측량단이고, 다른 하나는 인민무력부(군) 측지국이다. 그런데 진짜 좌표는 측지국이 갖고 있다. 중앙측량단의 좌표는 일부러 정확한 좌표와 다르게 작성한다. 좌표가 고급 비밀이기 때문에 외부에 정보가 새나가면 안된다는 이유 때문이다.

허 씨는 한국에 온 뒤 큰 허탈감을 느꼈다. 위성으로 GPS를 찍는 시대를 보니 그 무거운 장비를 메고 다니며 점 하나를 찍겠다고 며칠씩 바친 과거가 너무 허망하게 생각됐기 때문이다. 북한도 2005년경부터 러시아 위성항법체계인 ‘글로나스’의 도움을 받아 위성 측량을 시도했다고는 알려졌지만, 어느 정도 활용되는지는 아직 파악되지 않고 있다.

● 원시적 석탄채굴
허 씨는 2001년 말에 온성으로 돌아왔다. 뇌물이 없으면 이직도 힘든 세상이지만, 아버지도 없는데 어머니마저 아파서 쓰러졌다고 하니 집으로 보내준 것이다. 실제로 어머니를 간호할 사람은 허 씨 밖에 없었다.

새로 발령받은 직장은 풍인탄광 기술과였다. 그러나 할 일은 없었다. 1990년대 중반 고난의 행군 시기 온성 탄광들은 모두 침수됐다. 전기가 없어 물을 빼낼 수가 없었기 때문이다. 그렇게 몇 년을 방치하다보니 갱을 다시 사용할 수가 없었다.

탄광에 다니던 사람들은 먹고 살기 위해 다른 방법을 찾았다. 일제 때 했던 방식대로 얇은 탄층 채굴을 시작한 것이다. 삽과 곡괭이로 수직굴을 파들어 가는데, 운이 좋으면 8m에서 탄층을 만나기도 하지만, 운이 나쁘면 25m까지 파들어 간다. 탄층을 만나면, 그걸 따라 이번엔 가로로 파 들어간다. 탄층의 두께는 보통 0.8~1.2m였다. 양동이를 매단 도르래를 타고 수직굴에 들어가 몸을 돌리기도 어려운 좁은 공간에서 석탄을 파서 다시 양동이로 끌어올렸다. 도무지 현대인이라고 볼 수 없는 원시적 채탄방식이었지만, 그렇게 해서라도 석탄을 캐서 팔아야 장마당에서 먹을 것을 사올 수 있었다. 이런 수직굴을 온성에선 ‘노두’라고 불렀다. 온성에는 이런 노두가 수없이 많았다. 노두당 가족, 친척, 친구 등 5~7명이 팀을 이뤄 작업했는데, 이런 사람들을 ‘노두공’이라 불렀다. 북쪽은 날씨가 춥기 때문에 먹는 것 못지않게 석탄도 귀하다. 캐내면 파는 것은 큰 문제가 안됐다.

노두도 1년 내내 할 수 있는 것이 아니었다. 땅이 어는 1~3월에만 할 수 있었고, 봄이 와서 땅이 녹으면 수직갱이 버틸 수 없기 때문에 버려야 했다. 구멍을 방치해서도 안됐다. 단속기관 사람들이 찾아와 갱을 다시 메웠는지 조사한다. 단속할 권한이 있다는 것은 뇌물을 받을 수 있다는 것을 의미하기 때문에 얼렁뚱땅 넘어가지 않는다.

온성 전체에 이런 노두가 수없이 생겨났다 사라졌다. 겨울이면 할 일이 없는 농민들도 이 일에 매달렸다. 안전은 뒷전이라 붕괴와 추락, 가스질식 등으로 죽는 사람들이 계속 생겨났지만 큰 문제가 되진 않았다. 그 일을 하지 않으면 어차피 굶어죽기 때문이다.

허 씨가 소속된 기술과는 늘 각종 공사에 동원됐다. 도로 공사, 발전소 공사 등등 인력을 차출하는 공사는 끊이질 않았다. 김책공대에서 배운 지식을 쓸 곳은 없었다.

이렇게 살다가 어느새 결혼할 나이가 됐다. 그래도 허 씨는 명색이 탄광마을이 배출한 수재이고, 평양에서 김책공대까지 나왔던 터라 여성들에게 나름 인기가 있었다.

32살 때인 2006년 그는 10살 어린 여성과 결혼했다. 아내는 탄광 선전대 가수로 나름 인기가 좋았다. 이성적인 지식인과 감성적인 예술인의 조합이었다.

그렇지만 매력과 결혼생활은 별개였다. 결혼한 직후부터 둘은 다투는 일이 많았다. 서로가 서로를 견디기 어려워했다. 4년간의 결혼생활 끝에 둘은 이혼하기로 합의했다. 어린 딸은 아내가 키우기로 했다. 이때부터 그는 중국으로 눈을 돌렸다.


2022년 인천지하철 공사 현장에서 일하고 있는 허 씨.


● 두만강을 넘나들다
강 건너 연변 왕청에는 아버지의 삼촌과 사촌 등 친척들이 살고 있었다. 국경 근처라 많은 사람들이 중국을 드나들 때도 허 씨는 명색이 김책공대 졸업생인데 조국을 배반하면 안 된다는 마음이 컸다. 하지만 이혼 후 세상이 달리 보이기 시작했다. 이혼까지 한 마당에 눈치 볼 일도, 무서울 일도 없었다.

2010년 겨울 그는 국경경비대에 돈을 쥐어주고 두만강을 넘었다. 탈북한 것은 아니고, 친척에게 도움만 받자는 목적이었다. 저녁에 넘어가서 친척에게 전화를 하고 새벽이 되기 전에 다시 북으로 넘어왔다.

2012년 2월 그는 두 번째로 중국으로 갔다. 당시는 김정일이 사망한지 얼마 안됐던 때라 경계가 매우 심했다. 그는 북한군 군복을 입고 길을 떠났다. 군인은 초소에서 잘 단속하지 않았기 때문이다. 강을 따라 난 도로로 한참을 가다가 어둠이 내릴 때 바로 두만강을 넘었다. 중국 부락의 아무 집이나 들어가 왕청 친척에게 전화를 좀 하고 싶다고 했다. 전화를 하고 몇 시간 정도 머물렀는데 갑자기 공안이 들이닥쳤다. 집주인이 그를 도와주는 척하고는 신고를 했던 것이다. 알고 보니 중국은 그즈음부터 탈북자를 신고하면 포상금을 주는 제도를 운영하고 있었다. 특히 북한군 국경경비대가 수시로 건너와 사람까지 죽이며 노략질을 했기 때문에, 중국에서 북한군은 최고의 기피대상이었다.

탈북자들이 북송 전 대기하는 도문변방수용소에 끌려갔는데 며칠 뒤 보위부에서 차를 갖고 건너와 그를 싣고 갔다.

그렇지만 그는 예상과는 달리 20일 정도 가수감됐다가 석방됐다. 서류를 보니 중국에 친척이 다 있는 것도 확실하니, 도움 좀 받으려 넘어갔을 뿐이라는 그의 말이 인정됐다. 탈북은 배반이지만, 그의 경우엔 일탈 정도로 간주됐다.

보위부라고 해봐야 어차피 한동네 사는 아는 사람들이었는데, 그들은 지역이 배출한 인재의 경력을 망가뜨리고 싶지 않았는지 그다지 혹독하게 대하진 않았다.

하지만 허 씨 입장에선 아무 것도 이루지 못하고 잡혀오니 화가 났다. 보위부 감방에서 그는 강 건너편에서 일하다 잡혀왔다는 사람을 알게 됐다. 그는 자기가 일했던 중국 훈춘의 한 슈퍼의 이름을 알려주면서, 다시 건너가 자기 이름을 대면 사장이 고용해 줄 것이라고 했다.

감방에서 나온 허 씨는 다시 두만강을 넘었다. 세 번째 탈북이었다. 그는 감옥 동기가 알려준 슈퍼로 갔다. 왕청에 가지 않은 이유는 친척들은 별로 도와줄 의향이 없어 보였기 때문이다.
슈퍼는 중국과 나진선봉을 연결하는 도로 옆에 있었는데, 두만강 건너 허 씨네 동네가 약 10㎞ 밖에 빤히 바로 보였다. 허 씨가 내륙 깊이 들어가지 않은 이유는 딸을 데려오고 싶어서였다. 고향 가까이 있어야 집과 연락이 수월하다고 판단했다. 허 씨가 일하는 슈퍼에는 북한을 오가는 운전기사들이 수시로 드나들었다. 이들을 통해 그는 전처에게 휴대전화를 전달할 수 있었다.

● 자수해 감옥에 가다
그렇게 1년쯤 지났는데 사고가 생겼다. 중국 휴대전화를 받은 전처가 그걸 이용해 돈 벌려고 브로커 일을 하다가 보위부에 체포된 것이다. 전 남편마저 실종되니 보위부는 그녀에게 한국행을 기도했다는 누명을 씌웠다.

그 소식이 허 씨에게도 전달됐다. 그래도 4년 간 살았던 정도 있고, 어린 딸까지 있으니 모르는 척 할 수가 없었다. 그는 다시 북에 나가기 위해 자수하기로 결심했다. 자신이 나타나야 전 남편이 한국에 갔다는 누명을 벗길 수 있을 것 같았다.

슈퍼 주인은 전과자 출신이었는데, 그가 북으로 가겠다고 하자 감옥에서 나오는 자기의 비법을 전수해주었다. 폐 주변에 황산철을 주사하면 고열이 나고, 염증이 생긴다면서, 자신은 그 방법으로 병원에 호송됐다가 도망쳤다고 했다. 단 이 방법의 단점은 한 달 안에 황산철을 세척하지 못하면 생명을 잃을 수도 있다고 했다.

허 씨는 그의 조언에 따라 폐에 황산철을 네 군데나 주입했다. 그리고 스스로 공안에 자수한 뒤 2014년 4월 13일 북송됐다. 그런데 이번은 보위부도 호락호락하지 않았다. 불행하게도 그가 자수한 뒤 공안이 그의 숙소를 뒤져 소지품을 북송할 때 함께 보냈던 것이다. 그 속에는 한국 영사관, 한국인 전도사 등의 전화번호가 적힌 수첩이 있었다. 이게 화근이 됐다. 고문이 시작됐다.

중국 사장이 알려준 방법은 확실히 효과가 있었다. 폐가 곪아가기 시작했던 것이다. 염증으로 심한 열까지 나 거동을 못할 정도가 됐다. 보위부는 감방에서 죽이긴 싫었는지 그를 가석방으로 집에 보냈다.

그때가 5월 말이었다. 중국 사장이 당부한 폐를 씻어야 하는 한 달이 지난 것이다. 집에 가서 이러저런 치료를 해봤지만 점점 악화됐다. 이렇게 지내다간 죽을 것 같았다. 그는 황산철 주사를 맞은 지 네 달이 지난 8월 23일 다시 중국으로 넘어왔다. 이번엔 하얼빈에 들어가 식당에 취직해 치료에만 전념했다. 하지만 몸은 점점 더 쇠약해졌다.

몸을 운신하기 어려워지자 허 씨는 죽더라도 고향에 가서 죽겠다고 생각했다. 그해 11월말 그는 다시 연길에 왔다. 연길에서 혹시나 도움을 받지 않을까 싶어 처음으로 교회에 찾아갔는데, 거기서 집도 제공하고 치료도 해주었다. 12월 말 허 씨는 산소 호흡기에 의지하는 신세가 됐다. 쇼크도 수시로 왔다. 교회에선 한국에 가서 치료를 받아야 살 수 있다며 한국행선을 주선해주었다.


교회 목회자가 될 꿈을 꾸던 당시의 허 씨. 2017년 뉴욕에서 찍은 사진이다.


● 최초로 남북 측량기사 자격 모두 획득
2015년 2월 그는 여러 탈북민들과 함께 한국으로 떠났다. 동남아 정글에서 일행을 따라갈 수 없을 때 친한 동생이 그를 업고 산을 넘었다. 우여곡절 끝에 한국에 도착했지만 다음날 적십자병원에 실려갔다. 이곳에서 4개월 동안 치료를 받다보니 하나원도 나오지 못했다.

허 씨는 인천에 거주지가 배정됐다. 건강이 너무 악화돼 일을 할 수 없는 상태라 2016년말까지 병원에 다니며 통원치료를 받았다. 하나원을 나온 다른 사람들이 하나둘 취직해 일자리를 얻는 것을 보고 조급한 마음에 노량진에서 공무원 시험을 준비하기도 했지만, 건강이 악화돼 다시 병원에 실려 가기도 했다. 한국의 의술은 역시 대단했다. 2년이라는 오랜 시간이 걸리긴 했지만, 끝내 허 씨를 완치시켰다.

치료 받는 기간에 허 씨는 목회자의 길을 걸어볼까 싶어 1년 남짓 성경공부도 했고, 화장품 회사에 취직해 일도 해봤지만 적성에 맞지 않았다.

많은 고민 끝에 북에서 배운 측량기사 일을 다시 해보려 알아보니, 남쪽도 측량기사는 일도 힘들고 보상도 적었다. 그렇지만 적성에 맞지 않는 곳에서 시간을 낭비하는 것보다는 그래도 자신있는 분야를 파보자는 생각에 2018년 한 엔지니어링 회사에 취직했다. 취직은 했어도 아무런 건설기술인 등급도 받지 못했기 때문에 회사에서 가장 많은 나이임에도 허드레 일만 해야 했다. 연봉은 2300만 원이었다. 그는 앞으로 살아가려면 자격증이 필수라는 것을 깨달았다. 어떤 자격을 갖고 있고, 어떤 경력을 쌓는가에 따라 연봉도 결정됐다.

허 씨는 회사에 다니며 1년 동안은 토목계측에 대한 용어부터 수첩에 적어 기본적인 지식을 배웠다. 그리고 이듬해 대구과학대에 입학했다. 일도 하면서 대학 공부까지 하려니 야간반을 다닐 수밖에 없었는데, 전국의 측량 관련 야간반 중에 대구가 그나마 가까웠다.

가깝다고 해도 당시 일을 하던 포항의 건설현장에서 150㎞나 떨어져 있었다. 일주일에 서너번씩 왕복 300㎞를 달려 수업을 듣고 오는 일은 결코 쉽지 않았다.

돌아오는 길에 너무 피곤해 졸음 쉼터에서 쪽잠을 자기 일쑤였다. 잠깐 눈을 붙인다는 것이 해가 중천에 걸릴 때까지 잔적도 있다.

허 씨는 대학 공부와 병행해 자격증 시험도 준비했다. 3년 동안 주경야독의 삶을 끈질기게 이어나간 끝에 그는 2022년 대학을 우수한 성적으로 졸업하고, 동시에 측량 및 지형공간정보기사와 토목기사 자격증을 받을 수 있었다. 허 씨는 남과 북에서 측량기사 자격을 각각 받은 최초의 사례다.

이러한 자격증과 경력을 인정받아 현재 허 씨는 건설기술인협회에서 고급기술인으로 인정받고 있으며, 300억 원 규모 이상의 토목공사를 책임지고 할 수 있다.


2019년 여수에서 전력 케이블용 해저터널 공사장에서 작업하는 모습. 뒷모습이 보이는 사람이 허 씨이다.


● “배워야 살 수 있다”
허 씨는 김책공대에서 무려 8년이나 공부를 했지만 여기선 모든 것을 다시 배워야 했다고 말했다.

“남쪽에 오니 용어는 물론, 장비나 자재 등 모든 것이 북한과 달랐습니다. 비유하면 북에서 통나무로 집을 짓는 법을 배웠는데, 남쪽에 오니 시멘트로 아파트를 짓는 격이죠. 그럼 집 짓는 법을 처음부터 다시 배워야하죠. 물론 북한 김책공대 과정이 전혀 의미 없는 것은 아닙니다. 북한에선 가장 기본적인 것, 즉 공부하는 방법을 배운 것 같습니다.”

그는 남북은 통합성에서도 차이가 크다고 했다.

“여기는 한 가지를 하려면 열 가지를 알아야 합니다. 많은 것들이 서로 연결돼 있어 전체적으로 진행되죠. 반면 북한은 직무가 매우 세분화돼 있고, 맡겨진 것만 하면 됐어요.”

자격증과 경력을 쌓고 나니 연봉도 빠르게 올라갔다. 김책공대까지 입학했던 북한 시골 수재가 다시 자신의 두뇌와 재능을 발휘하는 데는 오랜 시간이 걸리지 않았다. 2300만 원에서 시작했던 연봉은 6년 만에 3배 이상 높아졌다.

그는 여기에 만족하지 않았다. 그는 올해 3월 다시 대구대 공간정보전문기술 석사과정에 입학했다. 토질 및 기초기술사 자격을 취득하기 위해서다. 다른 자격증과는 달리 이 자격은 받기가 매우 어렵다. 허 씨의 계획은 향후 5년 안에 석사와 함께 기술사 자격을 획득하는 것이다.

그렇다고 공부만 할 수는 없는 일. 요즘 허 씨는 마산만에서 스팀배관 부설을 위한 해저터널을 뚫는 작업을 하고 있다. 그의 직책은 공사차장. 터널 시공 품질을 총괄하는 중요한 자리다.

건설 현장은 매우 거칠고 다툼도 많다. 외국인 근로자들도 많아 의사소통의 문제도 많다. 많은 어려움이 있지만 허 씨는 포기하지 않고 한 걸음씩 나가고 있다.

“거친 현장이라서 그런지 북에서 왔다고 우습게 보는 사람들이 많아요. 아직도 저는 ‘나는 여전히 이방인이구나’라는 생각을 하고 있습니다. 그렇지만 실력은 남에게 뒤진다고 생각하지 않습니다. 여러 곳에서 이직 제안도 많이 받습니다.”

2023년 남북하나재단이 주관한 정착사례 발표대회에서 그는 최우수상인 통일부 장관상을 받았다. 하지만 허 씨는 지금은 정착의 첫 발자국을 뗀 것에 불과하다고 했다.

그는 자신의 이름으로 회사를 만들어 키우는 것을 다음 목표로 정했다. 그 목표가 달성되면 또 할 일이 있다. 나아가 통일이 되면 할 계획도 미리 생각해두었다.

“저는 북에 돌아가 여기서 배운 기술을 북에 전수할 겁니다. 한국은 토목 공사를 할 때 측량, 시공, 품질까지 다 알아야 합니다. 통일 되면 교통인프라를 다시 정리해야 할 것인데, 북한에서 대학을 다녔던 동창들은 통합성에 있어 크게 떨어집니다. 이런 것을 가르쳐야죠. 그리고 남북에서 모두 측량기사 자격을 받은 유일한 사람이니 다른 꿈도 있습니다. 남북공간정보 통합지도체계를 만드는 것도 중요한 목표입니다.”

꿈을 말할 때 그는 가장 행복한 표정이었다.'''

In [63]:
chunks = text_splitter.split_text(script)
len(chunks)

29

In [64]:
chunks[0]

'허수현은 한반도 최북단 탄광마을이 낳은 수재였다. 그는 고등학교 졸업 직전 북한 전체에서 700명에게만 수여하는 ‘7.15 최우등상’을 수상했다. 7.15최우등상은 김정일이 평양 남산고급중학교를 졸업한 날인 1960년 7월 15일을 기념해 1987년에 만들어진 상이다. 지금은 이 상이 특권층 자식들을 대학에 보내기 위한 발판이 돼 각종 비리와 뇌물로 얼룩져 있지만, 상이 제정된 초기 몇 년은 정말 공부 잘하는 사람에게만 수여됐다. 상을 받게 되면 곧바로 중앙급 대학에 진학할 수 있었다. 북한에선 고등중학교 졸업생 중 20% 미만이 대학이나 전문학교에 갈 수 있다는 것을 감안하면 엄청난 특혜였다. 허 씨도 김책공업종합대학(김책공대)에 입학해 8년이나 공부했다. 그리고 그때 배운 지식을 활용해 지금은 한반도 최남단인 경남 마산의 해저터널 공사장에서 시공품질을 관리하는 공사차장으로 일하고 있다. 김책공대 졸업생이 네 번의 탈북을 반복한 뒤 건강이 악화돼 남의 등에 업혀 동남아 정글을 넘어 한국까지 오게 되고, 이후 남과 북에서 동시에 측량기사 자격을 받은 최초의 기술자가 돼 해저터널 공사장에서 일하게 되기까지 삶의 과정은 결코 순탄하지 않았다. 2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.'

In [65]:
chunks[1]

'● 신분 상승의 꿈\n그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다.'

In [66]:
chunks[2]

'세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다. 온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다. 탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다. 부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다. 아버지가 사망하자 허 씨는 1년 정도 방황했다. 학교에도 나가지 않았다.'

1회성 RAG를 구축하고 싶다면 사람이 하는 것이 제일 좋긴 합니다. 하지만 사람이 하는 것이 힘들다면 LLM에게 전달해 문맥 단위로 잘라달라고 부탁하는 것도 좋은 방법입니다.

한국어 청킹은 Naver Cloud Clova Studio를 사용하는 것도 너무 괜찮습니다.

# VectorDB
문장을 벡터로 만들어 유사도를 비교할 수 있습니다. 이 때 특정 문서(또는 청크)를 벡터화 하여, 유사도를 검색할 수 있는 여러 VectorDB들에 대해 알아봅니다.

## ChromaDB 사용하기

In [67]:
!pip install chromadb

In [68]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)
chunks = text_splitter.create_documents([script])
len(chunks)

22

In [69]:
chunks[0]

Document(metadata={}, page_content='허수현은 한반도 최북단 탄광마을이 낳은 수재였다. 그는 고등학교 졸업 직전 북한 전체에서 700명에게만 수여하는 ‘7.15 최우등상’을 수상했다. 7.15최우등상은 김정일이 평양 남산고급중학교를 졸업한 날인 1960년 7월 15일을 기념해 1987년에 만들어진 상이다.\n\n지금은 이 상이 특권층 자식들을 대학에 보내기 위한 발판이 돼 각종 비리와 뇌물로 얼룩져 있지만, 상이 제정된 초기 몇 년은 정말 공부 잘하는 사람에게만 수여됐다. 상을 받게 되면 곧바로 중앙급 대학에 진학할 수 있었다. 북한에선 고등중학교 졸업생 중 20% 미만이 대학이나 전문학교에 갈 수 있다는 것을 감안하면 엄청난 특혜였다.\n\n허 씨도 김책공업종합대학(김책공대)에 입학해 8년이나 공부했다. 그리고 그때 배운 지식을 활용해 지금은 한반도 최남단인 경남 마산의 해저터널 공사장에서 시공품질을 관리하는 공사차장으로 일하고 있다. 김책공대 졸업생이 네 번의 탈북을 반복한 뒤 건강이 악화돼 남의 등에 업혀 동남아 정글을 넘어 한국까지 오게 되고, 이후 남과 북에서 동시에 측량기사 자격을 받은 최초의 기술자가 돼 해저터널 공사장에서 일하게 되기까지 삶의 과정은 결코 순탄하지 않았다.')

In [70]:
# 텍스트(청크)가 입력 되면 -> 임베딩을 통해 벡터화
from langchain_community.vectorstores import Chroma

chroma_db = Chroma.from_documents(chunks, OpenAIEmbeddings())
chroma_db

### 유사도 검색(similarity_search)
쿼리에 대한 유사도가 높은 문서를 검색해서 보여주는 역할

In [71]:
similar_docs = chroma_db.similarity_search("주인공은 어디서 태어났는가?")
print(similar_docs[0].page_content)

2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.


● 신분 상승의 꿈
그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.

온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.

탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.
부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.


### VectorStoreRetriver
`as_retreiver()` 메소드를 이용하면 말 그대로 "검색기"를 만들 수 있습니다. 유사도 검색과 마찬가지로 쿼리에 대한 유사도가 가장 높은 문서를 답변해 주지만, 검색에 관련된 여러 옵션을 부여하는 것이 가능하며, 추후 RAG에도 사용이 가능합니다.

In [72]:
retriver = chroma_db.as_retriever()

relevant_docs = retriver.get_relevant_documents("주인공은 어디서 태어났는가?")

/var/folders/zk/p59k4t_16mlgf68b5hxclv6m0000gn/T/ipykernel_43669/2304472909.py:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_docs = retriver.get_relevant_documents("주인공은 어디서 태어났는가?")


In [73]:
print(f"문서의 개수 : {len(relevant_docs)}")
print("=====검색 결과=====")
print(relevant_docs[0].page_content)

문서의 개수 : 4
=====검색 결과=====
2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.


● 신분 상승의 꿈
그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.

온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.

탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.
부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.


In [74]:
# 유사한 문서 최대 k개만 검색하기
retriver = chroma_db.as_retriever(search_kwargs={"k": 1})

relevant_docs = retriver.get_relevant_documents("주인공은 어디서 태어났는가?")

print(f"문서의 개수 : {len(relevant_docs)}")
print("=====검색 결과=====")
print(relevant_docs[0].page_content)

문서의 개수 : 1
=====검색 결과=====
2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.


● 신분 상승의 꿈
그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.

온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.

탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.
부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.


**search_type**
- 검색 알고리즘을 지정할 수 있습니다.

|타입|설명|
|:---|:---|
|similarity|기본값으로서 유사도만을 사용합니다.|
|mmr|쿼리와 관련된 항목을 검색하면서 동시에 내용의 중복을 최소화 합니다.|
|similarity_score_threshold|score_threshold 기준을 충족하는 유사도 문서가 반환됩니다.|

In [75]:
# retriever 생성. 일반 similarity 사용
retriever = chroma_db.as_retriever(search_kwargs={"k": 2})

# 유사도가 가장 높은 1개 문서를 검색
relevant_docs = retriever.get_relevant_documents("주인공은 어디서 태어났는가?")

print(f"문서의 개수: {len(relevant_docs)}")
print("[검색 결과]\n"
)
for i in range(len(relevant_docs)):
    print(relevant_docs[i].page_content)
    print("===" * 20)

문서의 개수: 2
[검색 결과]

2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.


● 신분 상승의 꿈
그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.

온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.

탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.
부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.
32살 때인 2006년 그는 10살 어린 여성과 결혼했다. 아내는 탄광 선전대 가수로 나름 인기가 좋았다. 이성적인 지식인과 감성적인 예술인의 조합이었다.

그렇지만 매력과 결혼생활은 별개였다. 결혼한 직후부터 둘은 다투는 일이 많았다. 서로가 서로를 견디기 어려워했다. 4년간의 결혼생활 끝에 둘은 이혼하기로 합의했다. 어린 딸은 아내가 키우기로 했다. 이때부터 그는 중국으로 눈을 돌렸다.


2022년 인천지하철 공사 현장에서 일하고 있는 허 씨.


● 두만강을 넘나들다
강 건너 연변 왕청에는 아버지의 삼촌과 사촌 등 친척들이 살고 있었다. 국경 근처라 많은 사람들이 중국을 드나들 때도 허 씨는 명색이 김책공대 졸업생인데 조국을 배반하면 안 된다는 마음이 컸다. 하지만 이혼 후 세상이 달리 보이기 시작했다. 이혼까지 한 마당에 눈치 볼 일도, 무서울 일도 없었다.

2010년 겨울 그는 국경경비대에 돈을 쥐어주고 두만강을 넘었다. 탈북한 것은 아니고, 친척에게 도움만 받자는 

In [76]:
# MMR 검색
#  유사한 문서를 찾되, 다양한 특성들을 최대한 반영하기 위해 중복된 특성을 띄지 않는 문서를 우선적으로 찾는다.

retriever = chroma_db.as_retriever(search_type="mmr", search_kwargs={'k':2})

relevant_docs = retriever.get_relevant_documents("주인공은 어디서 태어났는가?")

print(f"문서의 개수: {len(relevant_docs)}")
print("[검색 결과]\n"
)
for i in range(len(relevant_docs)):
    print(relevant_docs[i].page_content)
    print("===" * 20)

문서의 개수: 2
[검색 결과]

2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.


● 신분 상승의 꿈
그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.

온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.

탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.
부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.
“저는 졸업장을 받고 나니, 3점짜리 과목이 몇 개 있었어요. 저는 대학 내내 3점을 받은 적이 없거든요. 5점 만점에 3점은 낙제를 겨우 면한 수준인데, 대학 교무부에 찾아가 싸우지도 않았어요. 어차피 이 체제가 싫어서 탈북하려 결심한 마당에 3점이 대수냐고 생각했죠.”

“저도 졸업증에 받지 않은 3점들이 있었어요. 권력자 부모를 둔 학생들은 좋은 곳에 가기 위해 대학 교무부 직원들에게 뇌물을 주고 컴퓨터에서 점수를 바꾸었어요. 5점 최우등생과, 4점 우등생, 3점 보통생의 전체 비율을 바꿀 수 없으니 5점으로 조작하려면 누군가의 점수를 빼앗아야 했죠. 제일 힘이 없는 우리가 점수를 빼앗긴 거였죠.”

최우등을 하고도, 돈과 권력이 없으면 3점 졸업증을 받아야 했던 시대. 허 씨와 기자는 그 시대에 평양에서 대학을 다녔다. 기숙사에서 밥을 몇 숟가락만 주던 때라 대학 시절을 떠올리면 배고팠던 기억밖에 없다. 졸업증을 받은 것만으로도 기적이었다. 뇌물로 성적을 조작하고, 남의 

In [77]:
# 유사도 수치를 제한하여 검색하는 기법입니다.
#  아래 예시에서는 k:3으로 설정하여 3개의 문서를 찾기로 했지만 score_threshold: 0.735를 넘는 문서가 2개밖에 없기 때문에 2개의 결과만 확인됩니다.

retriever = chroma_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.735},
)
relevant_docs = retriever.get_relevant_documents("주인공은 어디서 태어났는가?")
print(f"문서의 개수: {len(relevant_docs)}")
print("[검색 결과]\n")
for i in range(len(relevant_docs)):
    print(relevant_docs[i].page_content)
    print("===" * 20)

문서의 개수: 2
[검색 결과]

2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.


● 신분 상승의 꿈
그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.

온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.

탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.
부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.
32살 때인 2006년 그는 10살 어린 여성과 결혼했다. 아내는 탄광 선전대 가수로 나름 인기가 좋았다. 이성적인 지식인과 감성적인 예술인의 조합이었다.

그렇지만 매력과 결혼생활은 별개였다. 결혼한 직후부터 둘은 다투는 일이 많았다. 서로가 서로를 견디기 어려워했다. 4년간의 결혼생활 끝에 둘은 이혼하기로 합의했다. 어린 딸은 아내가 키우기로 했다. 이때부터 그는 중국으로 눈을 돌렸다.


2022년 인천지하철 공사 현장에서 일하고 있는 허 씨.


● 두만강을 넘나들다
강 건너 연변 왕청에는 아버지의 삼촌과 사촌 등 친척들이 살고 있었다. 국경 근처라 많은 사람들이 중국을 드나들 때도 허 씨는 명색이 김책공대 졸업생인데 조국을 배반하면 안 된다는 마음이 컸다. 하지만 이혼 후 세상이 달리 보이기 시작했다. 이혼까지 한 마당에 눈치 볼 일도, 무서울 일도 없었다.

2010년 겨울 그는 국경경비대에 돈을 쥐어주고 두만강을 넘었다. 탈북한 것은 아니고, 친척에게 도움만 받자는 

## FAISS
FAISS는 문맥 검색(Semantic Search)를 도와주는 Meta에서 만든 라이브러리 입니다. cpu, gpu를 모두 지원한다.

In [78]:
!pip install faiss-cpu

In [79]:
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(chunks, OpenAIEmbeddings())

In [80]:
similar_docs = faiss_db.similarity_search("주인공은 어디서 태어났는가?")

print(f"문서의 개수 : {len(similar_docs)}")
print("=====검색 결과=====")
print(similar_docs[0].page_content)

문서의 개수 : 4
=====검색 결과=====
2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.


● 신분 상승의 꿈
그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.

온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.

탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.
부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.


In [81]:
# 데이터베이스를 검색기로 사용하기.
retriever = faiss_db.as_retriever()

In [82]:
docs = retriever.invoke("주인공은 어디서 태어났는가?")

print(docs[0].page_content)

2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.


● 신분 상승의 꿈
그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.

온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.

탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.
부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.


In [83]:
# similarity_search_with_score : 쿼리 문서 간의 거리 점수로 리턴. 낮을 수록 유사한 문서

docs_and_scores = faiss_db.similarity_search_with_score("주인공은 어디서 태어났는가?")
docs_and_scores[0]

(Document(metadata={}, page_content='2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.\n\n\n● 신분 상승의 꿈\n그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.\n\n온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.\n\n탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.\n부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.'),
 0.3339321)

In [84]:
# similarity_search_by_vector : 임베딩 벡터와 유사한 문서를 검색
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

# 질문(질의)을 임베딩 벡터로 변환
query = "주인공은 어디서 태어났는가?"
embedding_vector = embeddings.embed_query(query)

# 임베딩 벡터를 사용해서 유사도 검색을 수행하고, 문서와의 점수를 반환
docs_and_scores = faiss_db.similarity_search_by_vector(embedding_vector)
docs_and_scores[0]

Document(metadata={}, page_content='2021년 강원도 동해시 한 아파트 공사장에서 시공 측량 작업을 진행하고 있는 허 씨.\n\n\n● 신분 상승의 꿈\n그가 태어난 한반도 최북단 온성군은 오랜 역사를 가지고 있다. 세종 22년인 1440년에 김종서 장군이 이곳을 평정한 뒤 군을 설치하고 온성이라 부르기 시작했다.\n\n온성에는 평안남도 안주 탄전에 이은 북한 최대의 갈탄 탄전이 있다. 1980년대까지만 해도 온성에는 연간 50만 톤 이상의 갈탄을 생산하는 탄광이 여러 개 있었다. 허 씨는 이런 대형 탄광 중 하나인 주원 탄광마을에서 1974년에 태어났다.\n\n탄광에서 일하는 사람들은 대개 출신성분이 나빴다. 허 씨의 부친도 마찬가지였다. 부친은 1960년대 후반까지 중국에서 살다가 문화대혁명 등의 격변기를 거치며 북한으로 넘어왔다.\n부친은 늘 허 씨에게 “너는 공부를 잘해 꼭 신분 상승을 해야 한다”고 말했다. 허 씨가 13세 때 부친은 갱이 붕괴돼 사망했다. 탄광마을에선 자주 일어나는 일이었다.')

## Emsemble Retriever

Ensemble Retriever는 여러 retriever를 입력으로 받아 get_relavant_documents() 메소드의 결과를 앙상블하고, Reciprocal Rank Fusion 알고리즘을 기반으로 결과를 재 순위화 합니다.

서로 다른 알고리즘들의 장점을 활용하므로서 EnsembleRetriever는 단일 알고리즘보다 더 나은 성능을 달성할 수 있습니다.

가장 일반적인 패턴은 sparse retriever와 dense retriever를 결합하는 것인데, 이는 두 retriever의 장점이 상호 보완적이기 때문입니다. 이를 하이브리드 검색(Hybrid Search)라고도 합니다.

Sparse Retriever는 키워드를 기반으로 관련 문서를 찾는 데 효과적이며, Dense Retriever는 의미적 유사성을 기반으로 관련 문서를 찾는데 효과적입니다.


In [85]:
!pip install rank_bm25

In [86]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

In [87]:
# 비타민 별 섭취할 수 있는 음식 정보
doc_list_1 = [
    "비타민A : 당근, 시금치, 감자 등의 주황색과 녹색 채소에서 섭취할 수 있습니다.",
    "비타민B : 전곡물, 콩, 견과류, 육류 등 다양한 식품에서 찾을 수 있습니다.",
    "비타민C : 오렌지, 키위, 딸기, 브로콜리, 피망 등의 과일과 채소에 많이 들어 있습니다.",
    "비타민D : 연어, 참치, 버섯, 우유, 계란 노른자 등에 함유되어 있습니다.",
    "비타민E : 해바라기씨, 아몬드, 시금치, 아보카도 등에서 섭취할 수 있습니다.",
]

# 비타민 별 효능 정보
doc_list_2 = [
    "비타민A : 시력과 피부 건강을 지원합니다.",
    "비타민B : 에너지 대사와 신경계 기능을 돕습니다.",
    "비타민C : 면역 체계를 강화하고 콜라겐 생성을 촉진합니다.",
    "비타민D : 뼈 건강과 면역 체계를 지원합니다.",
    "비타민E : 항산화 작용을 통해 세포를 보호합니다.",
]

In [88]:
bm25_retriever = BM25Retriever.from_texts(
    doc_list_1,
    metadatas=[{"source" : 1}] * len(doc_list_1)
)

bm25_retriever.k = 1 # 검색 결과 개수를 1개로 제한

In [89]:
embeddings = OpenAIEmbeddings()

faiss_db = FAISS.from_texts(
    doc_list_2,
    embeddings,
    metadatas=[{"source": 2}] * len(doc_list_2)
)

faiss_retriever = faiss_db.as_retriever(search_kwargs={"k": 1})

In [90]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.6, 0.4],
    search_type="mmr"
)

In [91]:
ensemble_result = ensemble_retriever.get_relevant_documents("비타민A의 효능은?")
ensemble_result

[Document(metadata={'source': 1}, page_content='비타민E : 해바라기씨, 아몬드, 시금치, 아보카도 등에서 섭취할 수 있습니다.'),
 Document(metadata={'source': 2}, page_content='비타민A : 시력과 피부 건강을 지원합니다.')]

In [92]:
ensemble_result = ensemble_retriever.get_relevant_documents("시력에 좋은 비타민은?")
ensemble_result

[Document(metadata={'source': 1}, page_content='비타민E : 해바라기씨, 아몬드, 시금치, 아보카도 등에서 섭취할 수 있습니다.'),
 Document(metadata={'source': 2}, page_content='비타민A : 시력과 피부 건강을 지원합니다.')]

In [93]:
ensemble_result = ensemble_retriever.get_relevant_documents("비타민E는 어떻게 섭취하나요?")
ensemble_result

[Document(metadata={'source': 1}, page_content='비타민E : 해바라기씨, 아몬드, 시금치, 아보카도 등에서 섭취할 수 있습니다.'),
 Document(metadata={'source': 2}, page_content='비타민E : 항산화 작용을 통해 세포를 보호합니다.')]